In [1]:
"""
Compute Hamidieh-style compositional features from chemical formulas.

Reproduces the exact statistical definitions used in Hamidieh's
main_script_production_9.R (get_features / extract functions), restricted
to the 30 SHAP-selected features used in the PBSE thesis pipeline:

    upper_critical_field, coherence_length            (GL-derived, computed
                                                         separately, not here)
    vF_predicted_ms                                    (from the vF pipeline,
                                                         not here)
    + 27 compositional statistics drawn from:
        atomic_mass, fie, atomic_radius, Density,
        ElectronAffinity, FusionHeat, ThermalConductivity, Valence

Element property source: EXACTLY RECOVERED from Final_dataset_83.csv itself.
tc.RData (Hamidieh's saved element lookup table) could not be parsed in this
environment (R workspace with unsupported binary object types), so instead
of substituting a generic CRC-table (which gave imperfect correlations,
particularly for Valence and ElectronAffinity), each element's raw property
value was back-solved by least squares from the identity

    wtd_mean_X(compound) = sum_i( mole_fraction_i * X(element_i) )

using all 21,263 compounds in the dataset as simultaneous equations for the
86 unknown per-element values. The system is heavily overdetermined (21,263
equations, 86 unknowns) and every property reconstructs the dataset's
wtd_mean_X columns with exactly zero residual, confirming the recovered
table is Hamidieh's actual original lookup table, not an approximation.


Formula notes (all taken directly from the R get_features() function):
  - std_y      : POPULATION std, ddof=0                 -> sqrt(mean((x-mean(x))^2))
  - wtd_std_y  : sqrt(sum(p * (x - wtd_mean_y)^2))       -> centered on WEIGHTED mean
  - gmean_y    : exp(mean(log(abs(x))))                  -> abs() guards negative values
  - wtd_gmean_y: exp(sum(p * log(abs(x))))
  - entropy_y  : normalizes on abs(x)/sum(abs(x)), natural log
  - wtd_entropy_y: normalizes on (p*abs(x))/sum(p*abs(x)), natural log
  - range_y    : max(x) - min(x)                          (unweighted)
  - wtd_range_y: max(p*x) - min(p*x)                       (elementwise product,
                                                             NOT (p_max*x_max)-(p_min*x_min))
  - p          : mole fraction = stoichiometric_count / sum(stoichiometric_counts)

Note: the recovered ElectronAffinity values already include Hamidieh's +1.5
offset baked in (recovered directly from the dataset), so no additional
offset is applied here. The recovered AtomicRadius values already reflect
his La/Ce manual overrides for the same reason.
"""

import re
import numpy as np
import pandas as pd


# ════════════════════════════════════════════════════════════════
# 1. ELEMENT PROPERTY TABLE
#    Order: [atomic_mass, atomic_radius(pm), valence, fie(kJ/mol),
#            electron_affinity(kJ/mol, +1.5 offset already applied by
#            Hamidieh), thermal_conductivity(W/mK), density(kg/m3),
#            fusion_heat(kJ/mol)]
#
#    77 elements: EXACTLY recovered from Final_dataset_83.csv by
#    least-squares back-solve (see module docstring). Every wtd_mean_X
#    column in Final_dataset_83.csv reconstructs with zero residual using
#    these values.
#
#    9 elements (He, Ne, Ar, Kr, Xe, Pm, Po, At, Rn): NEVER appear among
#    the 21,263 UCI compounds, so the back-solve is mathematically
#    undetermined for them (would otherwise be 0/0). These 9 use converted
#    fallback values instead, sourced from the standard CRC-style table
#    used earlier in this project (external_validation.ipynb) and
#    converted into this table's exact units/conventions:
#      - fie, ElectronAffinity: eV -> kJ/mol (x 96.485332)
#      - ElectronAffinity: +1.5 kJ/mol offset applied afterward, matching
#        the offset already baked into the 77 recovered values (verified
#        against known elements, e.g. recovered Cl EA = 350.5 kJ/mol =
#        real 349.0 kJ/mol + 1.5)
#      - Density: g/cm3 -> kg/m3 (x 1000)
#      - Valence: bond-count convention (e.g. noble gases = 0), not the
#        group-number convention a generic periodic table would give
#    These 9 elements essentially never occur in real superconductor
#    formulas (noble gases, or the highly unstable/radioactive Pm, Po, At,
#    Rn), so this fallback is a safety net for malformed input rather
#    than something expected to matter in practice -- but it now returns
#    physically reasonable numbers instead of zero if one appears.
#    CAVEAT: noble gases have genuine Valence = 0 (bond-count convention).
#    If one ever appears in an input formula, gmean_Valence/entropy_Valence
#    /wtd_gmean_Valence/wtd_entropy_Valence will raise a RuntimeWarning
#    (log(0)), since no real superconductor contains a noble gas this has
#    not been patched with an artificial non-zero Valence; flagged here so
#    it isn't mistaken for a bug if ever triggered by malformed input.
# ════════════════════════════════════════════════════════════════

ELEMENTAL_DATA = {
    'H': [1.007940, 53.000000, 1.000000, 1311.300000, 74.300000, 0.180500, 0.089900, 0.558000],
    'He': [4.003000, 31.000000, 0.000000, 2372.284900, 1.500000, 0.151300, 0.178500, 0.000000],
    'Li': [6.941000, 167.000000, 1.000000, 519.900000, 61.100000, 85.000000, 535.000000, 3.000000],
    'Be': [9.012182, 112.000000, 2.000000, 898.800000, 1.500000, 190.000000, 1848.000000, 7.950000],
    'B': [10.811000, 87.000000, 3.000000, 800.200000, 28.200000, 27.000000, 2460.000000, 50.000000],
    'C': [12.010700, 67.000000, 4.000000, 1085.700000, 155.400000, 140.000000, 2260.000000, 105.000000],
    'N': [14.006700, 56.000000, 3.000000, 1401.500000, 8.500000, 0.025830, 1.251000, 0.360000],
    'O': [15.999400, 48.000000, 2.000000, 1313.100000, 142.500000, 0.026580, 1.429000, 0.222000],
    'F': [18.998403, 42.000000, 1.000000, 1680.000000, 329.500000, 0.027700, 1.696000, 0.260000],
    'Ne': [20.180000, 38.000000, 0.000000, 2080.706200, 1.500000, 0.049300, 0.900200, 0.340000],
    'Na': [22.989769, 190.000000, 1.000000, 495.600000, 54.300000, 140.000000, 968.000000, 2.600000],
    'Mg': [24.305000, 145.000000, 2.000000, 737.300000, 1.500000, 160.000000, 1738.000000, 8.700000],
    'Al': [26.981539, 118.000000, 3.000000, 577.200000, 44.000000, 235.000000, 2700.000000, 10.700000],
    'Si': [28.085500, 111.000000, 4.000000, 786.000000, 135.100000, 150.000000, 2330.000000, 50.200000],
    'P': [30.973762, 98.000000, 5.000000, 1011.200000, 72.500000, 0.236000, 1823.000000, 0.640000],
    'S': [32.065000, 87.000000, 6.000000, 999.000000, 201.500000, 0.205000, 1960.000000, 1.730000],
    'Cl': [35.453000, 79.000000, 5.000000, 1254.900000, 350.500000, 0.008900, 3.214000, 3.200000],
    'Ar': [39.948000, 71.000000, 0.000000, 1520.608800, 1.500000, 0.017700, 1.783700, 1.188000],
    'K': [39.098300, 243.000000, 1.000000, 418.500000, 49.900000, 100.000000, 856.000000, 2.330000],
    'Ca': [40.078000, 194.000000, 2.000000, 589.400000, 3.870000, 200.000000, 1550.000000, 8.540000],
    'Sc': [44.955912, 184.000000, 3.000000, 630.800000, 19.600000, 16.000000, 2985.000000, 16.000000],
    'Ti': [47.867000, 176.000000, 4.000000, 657.800000, 9.100000, 22.000000, 4507.000000, 18.700000],
    'V': [50.941500, 171.000000, 5.000000, 650.100000, 52.100000, 31.000000, 6110.000000, 22.800000],
    'Cr': [51.996100, 166.000000, 6.000000, 652.400000, 65.800000, 94.000000, 7190.000000, 20.500000],
    'Mn': [54.938045, 161.000000, 4.000000, 716.800000, 1.500000, 7.700000, 7470.000000, 13.200000],
    'Fe': [55.845000, 156.000000, 3.000000, 759.100000, 17.200000, 79.000000, 7874.000000, 13.800000],
    'Co': [58.933195, 152.000000, 4.000000, 758.100000, 65.200000, 100.000000, 8900.000000, 16.200000],
    'Ni': [58.693400, 149.000000, 2.000000, 736.200000, 113.500000, 91.000000, 8908.000000, 17.200000],
    'Cu': [63.546000, 145.000000, 2.000000, 745.000000, 119.900000, 400.000000, 8960.000000, 13.100000],
    'Zn': [65.380000, 142.000000, 2.000000, 905.800000, 1.500000, 120.000000, 7140.000000, 7.350000],
    'Ga': [69.723000, 136.000000, 3.000000, 578.700000, 30.400000, 29.000000, 5904.000000, 5.590000],
    'Ge': [72.630000, 125.000000, 4.000000, 760.000000, 120.500000, 60.000000, 5323.000000, 31.800000],
    'As': [74.921600, 114.000000, 5.000000, 946.200000, 79.500000, 50.000000, 5727.000000, 27.700000],
    'Se': [78.960000, 103.000000, 6.000000, 940.400000, 196.500000, 0.520000, 4819.000000, 5.400000],
    'Br': [79.904000, 94.000000, 5.000000, 1142.000000, 326.100000, 0.120000, 3120.000000, 5.800000],
    'Kr': [83.798000, 188.000000, 0.000000, 1350.794600, 1.500000, 0.009400, 3.749300, 1.638000],
    'Rb': [85.467800, 265.000000, 1.000000, 402.800000, 48.400000, 58.000000, 1532.000000, 2.190000],
    'Sr': [87.620000, 219.000000, 2.000000, 549.000000, 6.530000, 35.000000, 2630.000000, 8.000000],
    'Y': [88.905850, 212.000000, 3.000000, 615.400000, 31.100000, 17.000000, 4472.000000, 11.400000],
    'Zr': [91.224000, 206.000000, 4.000000, 659.700000, 42.600000, 23.000000, 6511.000000, 21.000000],
    'Nb': [92.906380, 198.000000, 5.000000, 663.600000, 87.600000, 54.000000, 8570.000000, 26.800000],
    'Mo': [95.960000, 190.000000, 6.000000, 684.800000, 73.400000, 139.000000, 10280.000000, 36.000000],
    'Tc': [98.000000, 183.000000, 6.000000, 702.200000, 54.500000, 51.000000, 11500.000000, 23.000000],
    'Ru': [101.070000, 178.000000, 6.000000, 710.300000, 102.800000, 120.000000, 12370.000000, 25.700000],
    'Rh': [102.905500, 173.000000, 6.000000, 719.500000, 111.200000, 150.000000, 12450.000000, 21.700000],
    'Pd': [106.420000, 169.000000, 4.000000, 803.500000, 55.200000, 71.000000, 12023.000000, 16.700000],
    'Ag': [107.868200, 165.000000, 1.000000, 730.500000, 127.100000, 430.000000, 10490.000000, 11.300000],
    'Cd': [112.411000, 161.000000, 2.000000, 867.200000, 1.500000, 96.000000, 8650.000000, 6.300000],
    'In': [114.818000, 156.000000, 3.000000, 558.000000, 30.400000, 82.000000, 7310.000000, 3.260000],
    'Sn': [118.710000, 145.000000, 4.000000, 708.200000, 108.800000, 67.000000, 7310.000000, 7.000000],
    'Sb': [121.760000, 133.000000, 5.000000, 833.300000, 104.700000, 24.000000, 6697.000000, 19.700000],
    'Te': [127.600000, 123.000000, 6.000000, 869.000000, 191.700000, 3.000000, 6240.000000, 17.500000],
    'I': [126.904470, 115.000000, 7.000000, 1008.300000, 296.700000, 0.449000, 4940.000000, 7.760000],
    'Xe': [131.293000, 108.000000, 0.000000, 1170.367100, 1.500000, 0.005600, 5.890000, 2.300000],
    'Cs': [132.905452, 298.000000, 1.000000, 375.500000, 47.000000, 36.000000, 1879.000000, 2.090000],
    'Ba': [137.327000, 253.000000, 2.000000, 502.500000, 15.450000, 18.000000, 3510.000000, 8.000000],
    'La': [138.905470, 195.000000, 3.000000, 541.100000, 49.500000, 13.000000, 6146.000000, 6.300000],
    'Ce': [140.116000, 185.000000, 4.000000, 540.100000, 51.500000, 11.000000, 6689.000000, 5.500000],
    'Pr': [140.907650, 247.000000, 4.000000, 526.600000, 51.500000, 13.000000, 6640.000000, 6.900000],
    'Nd': [144.242000, 206.000000, 3.000000, 531.500000, 51.500000, 17.000000, 7010.000000, 7.100000],
    'Pm': [145.000000, 205.000000, 3.000000, 538.581100, 49.742700, 17.900000, 7260.000000, 7.130000],
    'Sm': [150.360000, 238.000000, 3.000000, 540.100000, 51.500000, 13.000000, 7353.000000, 8.500000],
    'Eu': [151.964000, 231.000000, 3.000000, 546.900000, 51.500000, 14.000000, 5244.000000, 9.300000],
    'Gd': [157.250000, 233.000000, 3.000000, 594.200000, 51.500000, 11.000000, 7901.000000, 10.000000],
    'Tb': [158.925350, 225.000000, 3.000000, 569.000000, 51.500000, 11.000000, 8219.000000, 10.800000],
    'Dy': [162.500000, 228.000000, 3.000000, 567.000000, 51.500000, 11.000000, 8551.000000, 11.100000],
    'Ho': [164.930320, 226.000000, 3.000000, 574.000000, 51.500000, 16.000000, 8795.000000, 17.000000],
    'Er': [167.259000, 226.000000, 3.000000, 581.000000, 51.500000, 15.000000, 9066.000000, 19.900000],
    'Tm': [168.934210, 222.000000, 3.000000, 589.000000, 51.500000, 17.000000, 9320.000000, 16.800000],
    'Yb': [173.054000, 222.000000, 3.000000, 603.000000, 51.500000, 39.000000, 6570.000000, 7.700000],
    'Lu': [174.966800, 217.000000, 3.000000, 513.000000, 51.500000, 16.000000, 9841.000000, 22.000000],
    'Hf': [178.490000, 208.000000, 4.000000, 575.200000, 1.500000, 23.000000, 13310.000000, 25.500000],
    'Ta': [180.947880, 200.000000, 5.000000, 760.100000, 32.500000, 57.000000, 16650.000000, 36.000000],
    'W': [183.840000, 193.000000, 6.000000, 769.700000, 80.100000, 170.000000, 19250.000000, 35.000000],
    'Re': [186.207000, 188.000000, 7.000000, 759.100000, 16.000000, 48.000000, 21020.000000, 33.000000],
    'Os': [190.230000, 185.000000, 6.000000, 819.800000, 107.600000, 87.000000, 22590.000000, 31.000000],
    'Ir': [192.217000, 180.000000, 6.000000, 868.100000, 152.500000, 150.000000, 22560.000000, 26.000000],
    'Pt': [195.084000, 177.000000, 6.000000, 868.100000, 206.800000, 71.000000, 21450.000000, 20.000000],
    'Au': [196.966569, 174.000000, 5.000000, 889.300000, 224.300000, 320.000000, 19300.000000, 12.500000],
    'Hg': [200.590000, 171.000000, 2.000000, 1006.000000, 1.500000, 8.300000, 13534.000000, 2.290000],
    'Tl': [204.383300, 156.000000, 3.000000, 588.900000, 20.700000, 46.000000, 11850.000000, 4.200000],
    'Pb': [207.200000, 154.000000, 4.000000, 715.200000, 36.600000, 35.000000, 11340.000000, 4.770000],
    'Bi': [208.980400, 143.000000, 5.000000, 702.900000, 92.700000, 8.000000, 9780.000000, 10.900000],
    'Po': [209.000000, 135.000000, 2.000000, 812.117000, 184.822100, 20.000000, 9320.000000, 13.000000],
    'At': [210.000000, 127.000000, 1.000000, 926.259200, 271.658900, 2.000000, 7000.000000, 6.000000],
    'Rn': [222.000000, 120.000000, 0.000000, 1036.734900, 1.500000, 0.003600, 9.730000, 3.000000],
}

# Column order matches ELEMENTAL_DATA lists.
# 'fie' kept as Hamidieh's short name (first ionization energy).
PROP_NAMES = [
    'atomic_mass', 'atomic_radius', 'Valence',
    'fie', 'ElectronAffinity', 'ThermalConductivity',
    'Density', 'FusionHeat',
]

# ════════════════════════════════════════════════════════════════
# 2. FORMULA PARSER
# ════════════════════════════════════════════════════════════════

def parse_formula(formula):
    """
    Parse a chemical formula string into {element: stoichiometric_count}.

    Elements with a zero coefficient (e.g. 'Bi0' in
    'Bi0Mo0.33Cu2.67Sr2Y1O7.41') are dropped, matching Hamidieh's own data
    cleaning: his R script comments explicitly flag such zero-coefficient
    entries as invalid and remove them. Keeping them would make that
    element's mole fraction exactly 0, which is a mathematical singularity
    for entropy (log(0)).
    """
    pattern = r'([A-Z][a-z]?)(\d*\.?\d*)'
    matches = re.findall(pattern, str(formula))
    composition = {}
    for element, count in matches:
        if not element:
            continue
        if element not in ELEMENTAL_DATA:
            return None  # unknown / unsupported element
        count = float(count) if count else 1.0
        composition[element] = composition.get(element, 0.0) + count

    # Drop zero-coefficient elements (see docstring)
    composition = {e: c for e, c in composition.items() if c > 0}

    if not composition:
        return None
    return composition


# ════════════════════════════════════════════════════════════════
# 3. get_features() — EXACT port of the R function
# ════════════════════════════════════════════════════════════════

def get_features(y, p):
    """
    y : array of the raw elemental property values for the elements present
    p : array of mole fractions (same order as y), sums to 1

    Returns the 10 statistics in Hamidieh's fixed order:
    [mean, wtd_mean, gmean, wtd_gmean, entropy, wtd_entropy,
     range, wtd_range, std, wtd_std]
    """
    y = np.asarray(y, dtype=float)
    p = np.asarray(p, dtype=float)

    mean_y = np.mean(y)
    wtd_mean_y = np.sum(p * y)

    gmean_y = np.exp(np.mean(np.log(np.abs(y))))
    wtd_gmean_y = np.exp(np.sum(p * np.log(np.abs(y))))

    tmp = np.abs(y) / np.sum(np.abs(y))
    entropy_y = -np.sum(tmp * np.log(tmp))

    tmp_w = (p * np.abs(y)) / np.sum(p * np.abs(y))
    wtd_entropy_y = -np.sum(tmp_w * np.log(tmp_w))

    range_y = np.max(y) - np.min(y)
    wtd_range_y = np.max(p * y) - np.min(p * y)   # elementwise product, R-exact

    std_y = np.sqrt(np.mean((y - mean_y) ** 2))            # population std (ddof=0)
    wtd_std_y = np.sqrt(np.sum((y - wtd_mean_y) ** 2 * p))  # centered on WEIGHTED mean

    return {
        'mean': mean_y, 'wtd_mean': wtd_mean_y,
        'gmean': gmean_y, 'wtd_gmean': wtd_gmean_y,
        'entropy': entropy_y, 'wtd_entropy': wtd_entropy_y,
        'range': range_y, 'wtd_range': wtd_range_y,
        'std': std_y, 'wtd_std': wtd_std_y,
    }


# ════════════════════════════════════════════════════════════════
# 4. TOP-30 SHAP-SELECTED COMPOSITIONAL FEATURES
#    (excludes upper_critical_field, coherence_length, vF_predicted_ms,
#     which are GL / vF-pipeline outputs, not raw compositional stats)
# ════════════════════════════════════════════════════════════════

# Each entry: (output_column_name, property_name, statistic_key)
TOP30_COMPOSITIONAL_SPEC = [
    ('mean_atomic_mass',                 'atomic_mass',         'mean'),
    ('wtd_mean_atomic_mass',             'atomic_mass',         'wtd_mean'),
    ('gmean_atomic_mass',                'atomic_mass',         'gmean'),
    ('wtd_gmean_atomic_mass',            'atomic_mass',         'wtd_gmean'),
    ('entropy_atomic_mass',              'atomic_mass',         'entropy'),
    ('wtd_entropy_atomic_mass',          'atomic_mass',         'wtd_entropy'),
    ('range_atomic_mass',                'atomic_mass',         'range'),
    ('wtd_range_atomic_mass',            'atomic_mass',         'wtd_range'),
    ('std_atomic_mass',                  'atomic_mass',         'std'),
    ('wtd_std_atomic_mass',              'atomic_mass',         'wtd_std'),
    ('range_fie',                        'fie',                 'range'),
    ('wtd_range_fie',                    'fie',                 'wtd_range'),
    ('std_fie',                          'fie',                 'std'),
    ('wtd_std_fie',                      'fie',                 'wtd_std'),
    ('entropy_fie',                      'fie',                 'entropy'),
    ('wtd_entropy_fie',                  'fie',                 'wtd_entropy'),
    ('mean_fie',                         'fie',                 'mean'),
    ('wtd_mean_fie',                     'fie',                 'wtd_mean'),
    ('gmean_fie',                        'fie',                 'gmean'),
    ('wtd_gmean_fie',                    'fie',                 'wtd_gmean'),
    ('mean_atomic_radius',                 'atomic_radius',       'mean'), 
    ('wtd_mean_atomic_radius',             'atomic_radius',       'wtd_mean'),
    ('gmean_atomic_radius',                'atomic_radius',       'gmean'),
    ('wtd_gmean_atomic_radius',            'atomic_radius',       'wtd_gmean'),
    ('entropy_atomic_radius',              'atomic_radius',       'entropy'),
    ('wtd_entropy_atomic_radius',          'atomic_radius',       'wtd_entropy'),
    ('range_atomic_radius',                'atomic_radius',       'range'),
    ('wtd_range_atomic_radius',            'atomic_radius',       'wtd_range'),
    ('std_atomic_radius',                  'atomic_radius',       'std'),
    ('wtd_std_atomic_radius',              'atomic_radius',       'wtd_std'),
    ('mean_Density',                       'Density',             'mean'),
    ('wtd_mean_Density',                   'Density',             'wtd_mean'),
    ('gmean_Density',                      'Density',             'gmean'),
    ('wtd_gmean_Density',                  'Density',             'wtd_gmean'),
    ('entropy_Density',                    'Density',             'entropy'),
    ('wtd_entropy_Density',                'Density',             'wtd_entropy'),
    ('range_Density',                      'Density',             'range'),
    ('wtd_range_Density',                  'Density',             'wtd_range'),
    ('std_Density',                        'Density',             'std'),
    ('wtd_std_Density',                    'Density',             'wtd_std'),
    ('mean_ElectronAffinity',              'ElectronAffinity',    'mean'),
    ('wtd_mean_ElectronAffinity',          'ElectronAffinity',    'wtd_mean'),
    ('gmean_ElectronAffinity',             'ElectronAffinity',    'gmean'),
    ('wtd_gmean_ElectronAffinity',         'ElectronAffinity',    'wtd_gmean'),
    ('entropy_ElectronAffinity',           'ElectronAffinity',    'entropy'),
    ('wtd_entropy_ElectronAffinity',       'ElectronAffinity',    'wtd_entropy'),
    ('range_ElectronAffinity',                'ElectronAffinity',       'range'),
    ('wtd_range_ElectronAffinity',            'ElectronAffinity',       'wtd_range'),
    ('std_ElectronAffinity',                  'ElectronAffinity',       'std'),
    ('wtd_std_ElectronAffinity',              'ElectronAffinity',       'wtd_std'),
    ('mean_FusionHeat',                       'FusionHeat',             'mean'),
    ('wtd_mean_FusionHeat',                   'FusionHeat',             'wtd_mean'),
    ('gmean_FusionHeat',                      'FusionHeat',             'gmean'),
    ('wtd_gmean_FusionHeat',                  'FusionHeat',             'wtd_gmean'),
    ('entropy_FusionHeat',                    'FusionHeat',             'entropy'),
    ('wtd_entropy_FusionHeat',                'FusionHeat',             'wtd_entropy'),
    ('range_FusionHeat',                      'FusionHeat',             'range'),
    ('wtd_range_FusionHeat',                  'FusionHeat',             'wtd_range'),
    ('std_FusionHeat',                        'FusionHeat',             'std'),
    ('wtd_std_FusionHeat',                    'FusionHeat',             'wtd_std'),
    ('mean_ThermalConductivity',              'ThermalConductivity',   'mean'),
    ('wtd_mean_ThermalConductivity',          'ThermalConductivity',   'wtd_mean'),
    ('gmean_ThermalConductivity',             'ThermalConductivity',   'gmean'),
    ('wtd_gmean_ThermalConductivity',         'ThermalConductivity',   'wtd_gmean'),
    ('entropy_ThermalConductivity',           'ThermalConductivity',   'entropy'),
    ('wtd_entropy_ThermalConductivity',       'ThermalConductivity',   'wtd_entropy'),
    ('range_ThermalConductivity',             'ThermalConductivity',   'range'),
    ('wtd_range_ThermalConductivity',         'ThermalConductivity',   'wtd_range'),
    ('std_ThermalConductivity',               'ThermalConductivity',   'std'),
    ('wtd_std_ThermalConductivity',           'ThermalConductivity',   'wtd_std'),
    ('mean_Valence',                          'Valence',               'mean'),
    ('wtd_mean_Valence',                      'Valence',               'wtd_mean'),
    ('gmean_Valence',                         'Valence',               'gmean'),
    ('wtd_gmean_Valence',                     'Valence',               'wtd_gmean'),
    ('entropy_Valence',                       'Valence',               'entropy'),
    ('wtd_entropy_Valence',                   'Valence',               'wtd_entropy'),
    ('range_Valence',                         'Valence',               'range'),
    ('wtd_range_Valence',                     'Valence',               'wtd_range'),
    ('std_Valence',                           'Valence',               'std'),
    ('wtd_std_Valence',                       'Valence',               'wtd_std'),
]

# Sanity check: 80 compositional + 3 non-compositional (vF, xi, Hc2) = 83
assert len(TOP30_COMPOSITIONAL_SPEC) == 80, \
    f"Expected 80 compositional features, got {len(TOP30_COMPOSITIONAL_SPEC)}"

# Properties actually needed (skip computing stats for unused properties)
_NEEDED_PROPS = sorted(set(prop for _, prop, _ in TOP30_COMPOSITIONAL_SPEC))


def compute_composition_features(formula):
    """
    Compute the 80 SHAP-selected compositional statistics for one formula.
    Returns a dict {output_column_name: value}, or None if the formula
    could not be parsed (unknown element).
    """
    comp = parse_formula(formula)
    if comp is None:
        return None

    elements = list(comp.keys())
    counts = np.array([comp[e] for e in elements], dtype=float)
    p = counts / counts.sum()  # mole fractions

    # Compute all 10 stats once per needed property, then pick out the
    # specific (property, statistic) pairs the top-30 list actually uses.
    stats_by_prop = {}
    for prop in _NEEDED_PROPS:
        prop_idx = PROP_NAMES.index(prop)
        y = np.array([ELEMENTAL_DATA[e][prop_idx] for e in elements], dtype=float)
        stats_by_prop[prop] = get_features(y, p)

    out = {}
    for col_name, prop, stat_key in TOP30_COMPOSITIONAL_SPEC:
        out[col_name] = stats_by_prop[prop][stat_key]

    return out


# ════════════════════════════════════════════════════════════════
# 5. BATCH DRIVER
# ════════════════════════════════════════════════════════════════

def compute_features_for_dataframe(df, formula_col='material'):
    """
    df must have a column of chemical formula strings.
    Returns a new DataFrame with the 80 compositional feature columns,
    plus 'formula' and a 'parse_failed' flag for any row that couldn't
    be parsed (unknown element symbol).
    """
    rows = []
    failed_formulas = []

    for formula in df[formula_col]:
        feats = compute_composition_features(formula)
        if feats is None:
            failed_formulas.append(formula)
            rows.append({col: np.nan for col, _, _ in TOP30_COMPOSITIONAL_SPEC})
        else:
            rows.append(feats)

    out_df = pd.DataFrame(rows)
    out_df.insert(0, 'formula', df[formula_col].values)
    out_df['parse_failed'] = out_df['formula'].isin(failed_formulas)

    if failed_formulas:
        print(f"WARNING: {len(failed_formulas)} formulas failed to parse "
              f"(unknown element). Examples: {failed_formulas[:5]}")

    return out_df


def compute_composition_features_from_counts(element_counts):
    """
    Same as compute_composition_features(), but takes a dict/Series of
    {element_symbol: stoichiometric_count} directly instead of parsing a
    formula string. Use this when a pre-parsed stoichiometry table is
    available (e.g. unique_m.csv's per-element columns) -- it sidesteps
    regex parsing entirely and is the more robust path for a large batch.

    Zero-count elements should already be excluded (or will be filtered
    out here automatically).
    """
    elements = [e for e, c in element_counts.items()
                if e in ELEMENTAL_DATA and c and c > 0]
    if not elements:
        return None

    counts = np.array([element_counts[e] for e in elements], dtype=float)
    p = counts / counts.sum()

    stats_by_prop = {}
    for prop in _NEEDED_PROPS:
        prop_idx = PROP_NAMES.index(prop)
        y = np.array([ELEMENTAL_DATA[e][prop_idx] for e in elements], dtype=float)
        stats_by_prop[prop] = get_features(y, p)

    out = {}
    for col_name, prop, stat_key in TOP30_COMPOSITIONAL_SPEC:
        out[col_name] = stats_by_prop[prop][stat_key]
    return out


def compute_features_from_stoichiometry_table(stoich_df, element_cols=None):
    """
    Batch version using a wide stoichiometry table (one column per element,
    one row per compound), e.g. unique_m.csv from Hamidieh's R pipeline.

    stoich_df    : DataFrame, one row per compound
    element_cols : list of column names that are element symbols. If None,
                   auto-detects by intersecting stoich_df.columns with
                   ELEMENTAL_DATA keys.

    Returns a DataFrame of the 80 compositional features, row-aligned with
    stoich_df (same index / order), plus a 'parse_failed' flag.
    """
    if element_cols is None:
        element_cols = [c for c in stoich_df.columns if c in ELEMENTAL_DATA]

    rows = []
    failed_mask = []

    for _, row in stoich_df[element_cols].iterrows():
        counts = {e: row[e] for e in element_cols if row[e] > 0}
        feats = compute_composition_features_from_counts(counts)
        if feats is None:
            failed_mask.append(True)
            rows.append({col: np.nan for col, _, _ in TOP30_COMPOSITIONAL_SPEC})
        else:
            failed_mask.append(False)
            rows.append(feats)

    out_df = pd.DataFrame(rows, index=stoich_df.index)
    out_df['parse_failed'] = failed_mask

    n_failed = sum(failed_mask)
    if n_failed:
        print(f"WARNING: {n_failed} rows had no usable elements.")

    return out_df


def save_features_csv_from_formulas(
    output_path,
    input_csv,
    formula_col='name',
    label_col='Tc',
    label_out_col='critical_temp',
):
    """
    Compute the 27 SHAP-selected compositional features for an external
    dataset given as (formula, label) pairs -- e.g. the Stanev external
    validation set -- using the formula-string parser path.

    input_csv     : path to a CSV with a formula column and a label column
    formula_col   : name of the column holding chemical formula strings
                    (must use PROPER CASE element symbols, e.g. 'Ba0.4K0.6Fe2As2';
                    lowercase formulas will fail to parse)
    label_col     : name of the target column in input_csv (e.g. 'Tc')
    label_out_col : name to rename the label column to in the output
                    (default 'critical_temp', matching Final_dataset_83.csv)

    Rows with unparseable formulas (unknown element, or NaN formula/label)
    are dropped, with a warning printed showing the count and examples.
    """
    df_in = pd.read_csv(input_csv)
    n_before = len(df_in)

    df_in = df_in.dropna(subset=[formula_col, label_col]).reset_index(drop=True)
    n_after_na_drop = len(df_in)
    if n_before != n_after_na_drop:
        print(f"Dropped {n_before - n_after_na_drop} rows with missing "
              f"formula/label.")

    computed = compute_features_for_dataframe(df_in, formula_col=formula_col)

    # Attach label, drop failed-parse rows
    computed[label_out_col] = df_in[label_col].values
    n_failed = computed['parse_failed'].sum()
    if n_failed:
        failed_examples = computed.loc[computed['parse_failed'], 'formula'].head(5).tolist()
        print(f"WARNING: {n_failed} formulas failed to parse and were "
              f"dropped. Examples: {failed_examples}")
        computed = computed[~computed['parse_failed']].reset_index(drop=True)

    computed = computed.drop(columns=['parse_failed'])
    computed = computed.rename(columns={'formula': formula_col})

    # Reorder: formula, label, then the 27 features
    feature_cols = [c for c, _, _ in TOP30_COMPOSITIONAL_SPEC]
    computed = computed[[formula_col, label_out_col] + feature_cols]

    computed.to_csv(output_path, index=False)
    print(f"Saved {computed.shape[0]} rows x {computed.shape[1]} columns "
          f"-> {output_path}")

    return computed


def save_features_csv(
    output_path,
    stoich_path='unique_m - Copy.csv',
    labels_path='Final_dataset_83.csv',
):
    """
    Compute the 27 SHAP-selected compositional features for the full UCI
    dataset and save them to CSV, with 'material' and 'critical_temp'
    attached as identifier columns (row-order aligned with labels_path,
    since unique_m - Copy.csv and Final_dataset_83.csv share the same
    21,263-row order).

    output_path : where to write the resulting CSV
    stoich_path : path to unique_m.csv-style per-element stoichiometry table
    labels_path : path to the main dataset (for material name / critical_temp)
    """
    df_stoich = pd.read_csv(stoich_path)
    df_labels = pd.read_csv(labels_path)

    element_cols = [c for c in df_stoich.columns
                    if c not in ('material', 'critical_temp')]
    computed = compute_features_from_stoichiometry_table(
        df_stoich, element_cols=element_cols)

    computed.insert(0, 'material', df_labels['material'].values)
    computed.insert(1, 'critical_temp', df_labels['critical_temp'].values)

    n_failed = computed['parse_failed'].sum()
    computed = computed.drop(columns=['parse_failed'])

    computed.to_csv(output_path, index=False)
    print(f"Saved {computed.shape[0]} rows x {computed.shape[1]} columns "
          f"-> {output_path}")
    if n_failed:
        print(f"WARNING: {n_failed} rows had no usable elements.")

    return computed


if __name__ == '__main__':
    # ----------------------------------------------------------------
    # Formula-string self-test (sanity check on a few known compounds)
    # ----------------------------------------------------------------
    test_formulas = ['Re7Zr1', 'Y1Ba2Cu3O7', 'MgB2', 'Nb3Sn']
    test_df = pd.DataFrame({'material': test_formulas})
    result = compute_features_for_dataframe(test_df)
    pd.set_option('display.width', 200)
    pd.set_option('display.max_columns', 10)
    print("Formula-parser self-test:")
    print(result[['formula', 'mean_Density', 'wtd_mean_Valence',
                   'range_ThermalConductivity', 'parse_failed']])
    print()

    # ----------------------------------------------------------------
    # Full-dataset validation against Final_dataset_83.csv, if available.
    # Uses the stoichiometry-table path (unique_m.csv-style per-element
    # columns), which sidesteps regex parsing and is the recommended
    # route for the full 21,263-row UCI dataset.
    # ----------------------------------------------------------------
    import os
    stoich_path = 'unique_m - Copy.csv'
    actual_path = 'Final_dataset_83.csv'

    if os.path.exists(stoich_path) and os.path.exists(actual_path):
        from scipy.stats import pearsonr

        print("Validating against Final_dataset_83.csv ...")
        df_stoich = pd.read_csv(stoich_path)
        element_cols = [c for c in df_stoich.columns
                        if c not in ('material', 'critical_temp')]
        computed = compute_features_from_stoichiometry_table(
            df_stoich, element_cols=element_cols)

        actual = pd.read_csv(actual_path)
        cols_to_check = [c for c, _, _ in TOP30_COMPOSITIONAL_SPEC]

        print(f"\n{'Feature':<35}{'Pearson r':>12}{'Max abs err':>15}")
        print("-" * 62)
        for col in cols_to_check:
            r, _ = pearsonr(actual[col].values, computed[col].values)
            max_err = np.max(np.abs(actual[col].values - computed[col].values))
            print(f"{col:<35}{r:>12.6f}{max_err:>15.6f}")
    else:
        print("Final_dataset_83.csv / unique_m - Copy.csv not found; "
              "skipping full-dataset validation.")

Formula-parser self-test:
      formula  mean_Density  wtd_mean_Valence  range_ThermalConductivity  parse_failed
0      Re7Zr1   13765.50000          6.625000                   25.00000         False
1  Y1Ba2Cu3O7    4235.85725          2.076923                  399.97342         False
2        MgB2    2099.00000          2.666667                  133.00000         False
3       Nb3Sn    7940.00000          4.750000                   13.00000         False

Validating against Final_dataset_83.csv ...

Feature                               Pearson r    Max abs err
--------------------------------------------------------------
mean_atomic_mass                       1.000000       0.000000
wtd_mean_atomic_mass                   1.000000       0.000000
gmean_atomic_mass                      1.000000       0.000001
wtd_gmean_atomic_mass                  1.000000       0.000001
entropy_atomic_mass                    1.000000       0.000000
wtd_entropy_atomic_mass                1.000000     

In [2]:
# save_features_csv_from_formulas(
#     output_path='stanev_250_uci_proportional_v3.csv',
#     input_csv='stanev_250_uci_proportional_v3.csv',
#     formula_col='material',   # e.g. 'material' or 'name'
#     label_col='critical_temp',           # e.g. 'critical_temp' or 'Tc'
# )

# from compute_top30_features import save_features_csv_from_formulas, TOP50_COMPOSITIONAL_SPEC
save_features_csv_from_formulas(
    output_path='stanev_top30_features_83.csv',
    input_csv='stanev_250_uci_proportional_83.csv',
    formula_col='material',
    label_col='critical_temp',

)

Saved 250 rows x 82 columns -> stanev_top30_features_83.csv


,material,critical_temp,mean_atomic_mass,wtd_mean_atomic_mass,gmean_atomic_mass,...,wtd_entropy_Valence,range_Valence,wtd_range_Valence,std_Valence,wtd_std_Valence
0,Cs1V3Sn0.35Sb4.65,4.1,106.079238,99.273606,99.460724,...,0.894583,4.0,2.472222,1.639360,1.258195
1,Ba2Ca0.2Y0.8Zn0.06Cu2.94O6.45,31.0,68.539375,52.027185,56.498666,...,1.284221,1.0,1.026506,0.372678,0.245210
2,Bi0.75Pb0.25,8.0,208.090200,208.535300,208.088296,...,0.514653,1.0,2.750000,0.500000,0.433013
3,Ce0.1Nd1.9Cu1O4,2.4,90.975850,59.373571,67.327555,...,1.066026,2.0,1.085714,0.829156,0.488438
4,Nb(BRh)2,7.6,68.874293,68.874293,46.929940,...,1.060944,3.0,1.000000,1.247219,1.247219
...,...,...,...,...,...,...,...,...,...,...,...
245,Hf0.11Zr0.11Ti0.11Nb0.11Re0.56,4.4,119.338876,149.429532,106.159814,...,1.075204,3.0,3.480000,1.166190,1.394955
246,Sr0.125La1.675Eu0.2Cu1O4,125.0,91.606974,57.365080,71.589783,...,1.159628,1.0,1.107143,0.489898,0.442843
247,Nb3Pd,3.5,99.663190,96.284785,99.433882,...,0.514653,1.0,2.750000,0.500000,0.433013
248,Sn0.4Sb0.6,3.0,120.235000,120.540000,120.225328,...,0.646091,1.0,1.400000,0.500000,0.489898


In [ ]:
## Old version
# =============================================================================
#  External Validation — Stanev Dataset
#  Computes: family, subfamily, vF, coherence length (xi), upper critical field (Hc2)
#
#  Inputs  : stanev_recomputed.csv   ← your existing recomputed Stanev file
#                                       (must have 'material' and 'critical_temp'
#                                        columns plus the 81 Hamidieh features)
#            vF_hybrid_model.pkl     ← ML fallback model
#            vF_family_encoder.pkl   ← OneHotEncoder for family column
#
#  Outputs : stanev_with_GL_features.csv
# =============================================================================

import pandas as pd
import numpy as np
import re
import joblib
import warnings
warnings.filterwarnings('ignore')

# =============================================================================
# 1. LOAD DATA & MODELS
# =============================================================================
# ── adjust filename to match your actual Stanev CSV ──
df = pd.read_csv('stanev_top30_features_83.csv')


model   = joblib.load('vF_hybrid_model.pkl')
encoder = joblib.load('vF_family_encoder.pkl')

print(f"Stanev dataset shape : {df.shape}")
print(f"Columns sample       : {df.columns.tolist()[:5]} ...")
print(f"Tc range             : {df['critical_temp'].min():.2f} – "
      f"{df['critical_temp'].max():.2f} K\n")

# =============================================================================
# 2. CLASSIFICATION FUNCTIONS
#    Copied exactly from fermi_velocity_169_fixed_1.ipynb
# =============================================================================
def get_subfamily(mat_orig):
    if not isinstance(mat_orig, str):
        return 'other'
    mat = mat_orig.replace(" ", "")

    def has(element, string):
        return re.search(f"{element}(?![a-z])", string) is not None

    def has_major(element, string, threshold=0.5):
        m = re.search(rf"{element}(?![a-z])([0-9]*\.?[0-9]*)", string)
        if m is None:
            return False
        num = m.group(1)
        return True if num == '' else float(num) >= threshold

    if has('Cu', mat) and has('O', mat):
        if has('Mo', mat) and any(has(x, mat) for x in ['S', 'Se']):
            return 'chalcogenide'

    if has('Mg', mat) and has('B', mat):
        return 'MgB2_type'

    if has('Ce', mat) and has('In', mat) and not has('O', mat):
        return 'heavy_fermion_115'

    if has('Cu', mat) and has('O', mat):
        if has('Ce', mat) and has('In', mat):
            return 'heavy_fermion_115'
        if has('Ce', mat) and any(has(r, mat) for r in ['Nd', 'Pr', 'Sm', 'La']):
            return 'cuprate_electron_doped'
        if has('La', mat) and not any(has(x, mat) for x in ['Y', 'Bi', 'Tl', 'Hg']):
            return 'cuprate_LSCO'
        if has('Bi', mat) and has('Sr', mat):
            return 'cuprate_BSCCO'
        RE = ['Er', 'Gd', 'Sm', 'Nd', 'Ho', 'Dy', 'Eu', 'Lu', 'Pr', 'Tb', 'Tm', 'Yb']
        if has('Ba', mat) and (has('Y', mat) or any(has(r, mat) for r in RE)):
            return 'cuprate_YBCO'
        if has('Tl', mat):
            return 'cuprate_Tl_based'
        if has('Hg', mat):
            return 'cuprate_Hg_based'
        return 'cuprate_other'

    if has_major('Fe', mat):
        if has('As', mat) and has('O', mat):
            return 'iron_1111'
        if (has('Te', mat) or has('Se', mat)) and not has('As', mat):
            return 'iron_11'
        if has('As', mat) and any(has(x, mat) for x in ['Ba', 'Sr', 'Ca', 'K', 'Eu']):
            return 'iron_122'
        if has('As', mat) and (has('Li', mat) or has('Na', mat)):
            return 'iron_111'
        if has('As', mat) or has('P', mat):
            return 'iron_other'
        return 'iron_other'

    if has('Ce', mat) and any(has(x, mat) for x in
                               ['Pt', 'Pd', 'Rh', 'Co', 'In', 'Ru', 'Cu', 'Si']):
        return 'heavy_fermion_Ce'
    if has('Yb', mat) and any(has(x, mat) for x in ['Rh', 'Pt', 'Ni']):
        return 'heavy_fermion_Yb'

    RE2 = ['Lu', 'Ho', 'Er', 'Dy', 'Tm', 'Gd', 'Y', 'Tb']
    if has('Ni', mat) and has('B', mat) and has('C', mat) and \
       any(has(r, mat) for r in RE2):
        return 'borocarbide'

    if has('Nb', mat) and any(has(x, mat) for x in ['Sn', 'Al', 'Ge']) \
       and not has_major('Fe', mat):
        return 'conventional_A15'
    if has('V', mat) and any(has(x, mat) for x in ['Si', 'Ga']) \
       and not has_major('Fe', mat):
        return 'conventional_A15'

    if has('Nb', mat):
        return 'conventional_NbTi'

    if any(has(x, mat) for x in ['Se', 'Te', 'S']) and not has_major('Fe', mat):
        return 'chalcogenide'

    if has('Mg', mat) and has('B', mat):
        return 'MgB2_type'

    if (any(has(x, mat) for x in ['Pb', 'Al', 'Sn', 'In'])
            and not has_major('Fe', mat)
            and not has('Cu', mat)):
        return 'conventional_simple_metal'

    return 'other'


def get_family(mat_orig):
    sf = get_subfamily(mat_orig)
    family_map = {
        'cuprate_LSCO':              'Cuprates',
        'cuprate_YBCO':              'Cuprates',
        'cuprate_BSCCO':             'Cuprates',
        'cuprate_Tl_based':          'Cuprates',
        'cuprate_Hg_based':          'Cuprates',
        'cuprate_electron_doped':    'Cuprates',
        'cuprate_other':             'Cuprates',
        'iron_1111':                 'Iron-based',
        'iron_122':                  'Iron-based',
        'iron_111':                  'Iron-based',
        'iron_11':                   'Iron-based',
        'iron_other':                'Iron-based',
        'MgB2_type':                 'Mg-B related',
        'conventional_A15':          'Conventional / Alloys',
        'conventional_NbTi':         'Conventional / Alloys',
        'conventional_simple_metal': 'Conventional / Alloys',
        'borocarbide':               'Conventional / Alloys',
        'chalcogenide':              'Chalcogenide',
        'heavy_fermion_Ce':          'Heavy Fermion',
        'heavy_fermion_Yb':          'Heavy Fermion',
        'heavy_fermion_115':         'Heavy Fermion',
        'other':                     'Conventional / Alloys',
    }
    return family_map.get(sf, 'Conventional / Alloys')


# =============================================================================
# 3. vF LOOKUP TABLE (same canonical values as your training pipeline)
# =============================================================================
VF_LOOKUP = {
    'cuprate_LSCO':              245000,
    'cuprate_YBCO':              273400,
    'cuprate_BSCCO':             304000,
    'cuprate_Tl_based':          320000,
    'cuprate_Hg_based':          290000,
    'cuprate_electron_doped':    230000,
    'cuprate_other':             273400,
    'iron_1111':                 138000,
    'iron_122':                  121500,
    'iron_111':                  109000,
    'iron_11':                    95000,
    'iron_other':                121500,
    'MgB2_type':                 560000,
    'conventional_A15':          850000,
    'conventional_NbTi':         760000,
    'conventional_simple_metal': 1800000,
    'borocarbide':               165000,
    'chalcogenide':               95000,
    'heavy_fermion_Ce':            7595,
    'heavy_fermion_Yb':            5000,
    'heavy_fermion_115':           7595,
    'other':                     760000,
}

AMBIGUOUS_SUBFAMILIES = {'other', 'iron_other', 'cuprate_other'}

SINGLE_VALUE_FAMILIES = {
    'Chalcogenide': 95000,
    'Mg-B related': 560000,
}

VALID_VF = np.array(sorted(set(VF_LOOKUP.values())))


def snap_to_valid(pred_ms_array):
    pred_ms_array = np.asarray(pred_ms_array).reshape(-1, 1)
    diffs = np.abs(pred_ms_array - VALID_VF)
    return VALID_VF[diffs.argmin(axis=1)]


def apply_family_override(pred_ms_array, family_array):
    pred_ms_array = np.asarray(pred_ms_array).copy()
    family_array  = np.asarray(family_array)
    for fam, val in SINGLE_VALUE_FAMILIES.items():
        pred_ms_array[family_array == fam] = val
    return pred_ms_array


# =============================================================================
# 4. ASSIGN FAMILY & SUBFAMILY
# =============================================================================
# material column — adjust name if yours is different (e.g. 'formula', 'name')
MATERIAL_COL = 'material'

df['subfamily'] = df[MATERIAL_COL].apply(get_subfamily)
df['family']    = df[MATERIAL_COL].apply(get_family)
df['is_ambiguous'] = df['subfamily'].isin(AMBIGUOUS_SUBFAMILIES)

print("Family distribution (Stanev):")
print(df['family'].value_counts().to_string())
print(f"\nSubfamily distribution:")
print(df['subfamily'].value_counts().to_string())
print(f"\nAmbiguous rows : {df['is_ambiguous'].sum()} "
      f"({df['is_ambiguous'].mean()*100:.1f}%)")

# =============================================================================
# 5. PREDICT vF — hybrid: direct lookup + ML fallback
# =============================================================================
NUMERIC_FEATURES = [
    "wtd_std_fie",
    "wtd_entropy_ElectronAffinity",
    "wtd_std_ThermalConductivity",
    "wtd_entropy_fie",
    "wtd_std_FusionHeat",
    "range_Valence",
    "mean_ElectronAffinity",
    "wtd_std_Valence",
    "wtd_mean_FusionHeat",
    "wtd_entropy_FusionHeat",
    "wtd_gmean_Density",
    "wtd_mean_Valence",
]

# check all required numeric features are present
missing = [f for f in NUMERIC_FEATURES if f not in df.columns]
if missing:
    print(f"\nWARNING: Missing features for ML fallback: {missing}")
    print("These must be present in your recomputed Stanev CSV.")
    print("ML fallback rows will default to subfamily lookup value.\n")

vf_pred = np.zeros(len(df))
pred_source = np.array(['lookup'] * len(df), dtype=object)

# --- non-ambiguous: direct lookup ---
lookup_mask = ~df['is_ambiguous'].values
vf_pred[lookup_mask] = df.loc[lookup_mask, 'subfamily'].map(VF_LOOKUP).values

# --- ambiguous: ML fallback ---
amb_mask = df['is_ambiguous'].values
if amb_mask.sum() > 0 and len(missing) == 0:
    fam_enc  = encoder.transform(df.loc[amb_mask, ['family']])
    X_num    = df.loc[amb_mask, NUMERIC_FEATURES].values
    X_amb    = np.hstack([X_num, fam_enc])
    raw_log  = model.predict(X_amb)
    raw_ms   = 10 ** raw_log
    snapped  = snap_to_valid(raw_ms)
    snapped  = apply_family_override(snapped, df.loc[amb_mask, 'family'].values)
    vf_pred[amb_mask]     = snapped
    pred_source[amb_mask] = 'model'
elif amb_mask.sum() > 0 and len(missing) > 0:
    # fallback to 'other' lookup value when features missing
    vf_pred[amb_mask] = VF_LOOKUP['other']
    pred_source[amb_mask] = 'lookup_default'

df['vF_predicted_ms']   = vf_pred.astype(int)
df['prediction_source'] = pred_source

print(f"\nvF prediction source breakdown:")
print(df['prediction_source'].value_counts().to_string())
print(f"\nvF range: {df['vF_predicted_ms'].min():,} – "
      f"{df['vF_predicted_ms'].max():,} m/s")

# =============================================================================
# 6. COMPUTE GL FEATURES
# =============================================================================
Kb    = 1.380649e-23      # J/K
Pi    = np.pi
Phi0  = 2.067833848e-15   # Wb
h_cut = 1.054571817e-34   # J·s

Tc = np.clip(df['critical_temp'].values, 0.01, None)
vF = df['vF_predicted_ms'].values

energy_gap = 1.764 * Kb * Tc
df['coherence_length']     = (h_cut * vF) / (Pi * energy_gap)   # metres
df['upper_critical_field'] = Phi0 / (2 * Pi * df['coherence_length'] ** 2)  # Tesla

print(f"\nGL features computed:")
print(f"  xi  range : {df['coherence_length'].min():.3e} – "
      f"{df['coherence_length'].max():.3e} m")
print(f"  Hc2 range : {df['upper_critical_field'].min():.3f} – "
      f"{df['upper_critical_field'].max():.3f} T")

# =============================================================================
# 7. SANITY CHECK — family-level vF summary
# =============================================================================
print("\nPer-family vF summary:")
summary = df.groupby('family')['vF_predicted_ms'].agg(
    n='count', mean='mean', std='std', min='min', max='max'
).round(0)
print(summary.to_string())


In [ ]:
## New version
# =============================================================================
#  External Validation — Stanev Dataset
#  Computes: family, subfamily, vF, coherence length (xi), upper critical field (Hc2)
#
#  Inputs  : stanev_recomputed.csv   ← your existing recomputed Stanev file
#                                       (must have 'material' and 'critical_temp'
#                                        columns plus the 81 Hamidieh features)
#            vF_hybrid_model.pkl     ← ML fallback model
#            vF_family_encoder.pkl   ← OneHotEncoder for family column
#
#  Outputs : stanev_with_GL_features.csv
# =============================================================================

import pandas as pd
import numpy as np
import re
import joblib
import warnings
warnings.filterwarnings('ignore')

# =============================================================================
# 1. LOAD DATA & MODELS
# =============================================================================
# ── adjust filename to match your actual Stanev CSV ──
df = pd.read_csv('stanev_top30_features_83.csv')


model   = joblib.load('vF_hybrid_model.pkl')
encoder = joblib.load('vF_family_encoder.pkl')

print(f"Stanev dataset shape : {df.shape}")
print(f"Columns sample       : {df.columns.tolist()[:5]} ...")
print(f"Tc range             : {df['critical_temp'].min():.2f} – "
      f"{df['critical_temp'].max():.2f} K\n")

# =============================================================================
# 2. CLASSIFICATION FUNCTIONS
#    Copied exactly from fermi_velocity_169_fixed_1.ipynb
# ============================================================================

def get_subfamily(mat_orig):
    if not isinstance(mat_orig, str):
        return 'other'
    mat = mat_orig.replace(" ", "")

    def has(element, string):
        # element NOT followed by a lowercase letter -> avoids Cu matching 'ca' etc.
        return re.search(f"{element}(?![a-z])", string) is not None

    def has_major(element, string, threshold=0.5):
        m = re.search(rf"{element}(?![a-z])([0-9]*\.?[0-9]*)", string)
        if m is None:
            return False
        num = m.group(1)
        return True if num == '' else float(num) >= threshold

    if has('Cu', mat) and has('O', mat):
        # Exclude Chevrel phases: Cu-Mo-S/Se compounds
        if has('Mo', mat) and any(has(x, mat) for x in ['S', 'Se']):
            return 'chalcogenide'
    # ── CUPRATES (Cu + O) ──────────────────────────────────────────────────
    # MgB2 BEFORE Fe block
    if has('Mg', mat) and has('B', mat):
        return 'MgB2_type'

    # heavy_fermion_115 BEFORE cuprate block, no-O guard
    if has('Ce', mat) and has('In', mat) and not has('O', mat):
        return 'heavy_fermion_115'

    # CUPRATES (Cu + O)
    if has('Cu', mat) and has('O', mat):
        RE = ['Er', 'Gd', 'Sm', 'Nd', 'Ho', 'Dy', 'Eu', 'Lu', 'Pr', 'Tb', 'Tm', 'Yb']

        if has('Ce', mat) and has('In', mat) and not has('O', mat):
            return 'heavy_fermion_115'
        if has('Ce', mat) and any(has(r, mat) for r in ['Nd', 'Pr', 'Sm', 'La']):
            return 'cuprate_electron_doped'
        if has('La', mat) and not any(has(x, mat) for x in ['Y', 'Bi', 'Tl', 'Hg']) and not has_major('Ba', mat):
            return 'cuprate_LSCO'
        if has('Bi', mat) and has('Sr', mat) and not has('Hg', mat):
            return 'cuprate_BSCCO'
        if has('Ba', mat) and (has('Y', mat) or has('La', mat) or any(has(r, mat) for r in RE)):
            return 'cuprate_YBCO'
        # Tl before Hg
        if has('Tl', mat):
            return 'cuprate_Tl_based'
        if has('Hg', mat):
            return 'cuprate_Hg_based'
        return 'cuprate_other'


    # ── IRON-BASED (Fe) ───────────────────────────────────────────────────
    if has_major('Fe', mat):
        if has('Ca', mat) and has('Pt', mat) and has('As', mat):
            return 'iron_other'
        if has('As', mat) and (has('O', mat) or has('H', mat)):
            return 'iron_1111'
        if any(has(a, mat) for a in ['K', 'Rb', 'Cs', 'Tl', 'Na']) and has('Se', mat) and not has('As', mat):
            return 'iron_245'

        if (has('Te', mat) or has('Se', mat)) and not has('As', mat):
            return 'iron_11'
        # 122 before 111
        if has('As', mat) and any(has(x, mat) for x in ['Ba','Sr','Ca','K','Eu']):
            return 'iron_122'
        if has('As', mat) and (has('Li', mat) or has('Na', mat)):
            return 'iron_111'
        if has('As', mat) or has('P', mat):
            return 'iron_other'
        return 'iron_other'

    # ── HEAVY FERMION ─────────────────────────────────────────────────────
    if has('Ce', mat) and any(has(x, mat) for x in ['Pt', 'Pd', 'Rh', 'Co', 'In', 'Ru', 'Cu', 'Si']):
        return 'heavy_fermion_Ce'
    if has('Yb', mat) and any(has(x, mat) for x in ['Rh', 'Pt', 'Ni']):
        return 'heavy_fermion_Yb'

    # ── BOROCARBIDE  (must come BEFORE generic C check) ───────────────────
    RE2 = ['Lu', 'Ho', 'Er', 'Dy', 'Tm', 'Gd', 'Y', 'Tb']
    if has('Ni', mat) and has('B', mat) and has('C', mat) and any(has(r, mat) for r in RE2):
        return 'borocarbide'

    # ── CONVENTIONAL A15 ──────────────────────────────────────────────────
    if has('Nb', mat) and any(has(x, mat) for x in ['Sn', 'Al', 'Ge']) and not has_major('Fe', mat):
        return 'conventional_A15'

    if has('V', mat) and any(has(x, mat) for x in ['Si', 'Ga', 'Sn', 'In', 'Al']) and not has_major('Fe', mat):
        return 'conventional_A15'

    # ── CONVENTIONAL Nb-BASED (RENAMED, CHANGE 9) ───────────────────────────
    if has('Nb', mat):
        return 'conventional_Nb_based'

    # ── CHALCOGENIDE (non-Fe) ─────────────────────────────────────────────
    if any(has(x, mat) for x in ['Se', 'Te', 'S']) and not has_major('Fe', mat):
        return 'chalcogenide'

    # ── MgB2-TYPE ─────────────────────────────────────────────────────────
    if has('Mg', mat) and has('B', mat):
        return 'MgB2_type'

    # ── BISMUTHATE (NEW, CHANGE 7) ──────────────────────────────────────────
    if has('Ba', mat) and has('Bi', mat) and has('O', mat) and any(has(x, mat) for x in ['Pb', 'K']):
        return 'other'

    # ── SIMPLE METALS ─────────────────────────────────────────────────────
    if (any(has(x, mat) for x in ['Pb', 'Al', 'Sn', 'In'])
            and not has_major('Fe', mat)
            and not has('Cu', mat)):
        return 'conventional_simple_metal'

    return 'other'


def get_family(mat_orig):
    sf = get_subfamily(mat_orig)
    family_map = {
        'cuprate_LSCO':              'Cuprates',
        'cuprate_YBCO':              'Cuprates',
        'cuprate_BSCCO':             'Cuprates',
        'cuprate_Tl_based':          'Cuprates',
        'cuprate_Hg_based':          'Cuprates',
        'cuprate_electron_doped':    'Cuprates',
        'cuprate_other':             'Cuprates',
        'iron_1111':                 'Iron-based',
        'iron_122':                  'Iron-based',
        'iron_111':                  'Iron-based',
        'iron_11':                   'Iron-based',
        'iron_245':                  'Iron-based',   # NEW
        'iron_other':                'Iron-based',
        'MgB2_type':                 'Mg-B related',
        'conventional_A15':          'Conventional / Alloys',
        'conventional_Nb_based':     'Conventional / Alloys',
        'conventional_simple_metal': 'Conventional / Alloys',
        'borocarbide':               'Conventional / Alloys',
        'chalcogenide':              'Chalcogenide',
        'heavy_fermion_Ce':          'Heavy Fermion',
        'heavy_fermion_Yb':          'Heavy Fermion',
        'heavy_fermion_115':         'Heavy Fermion',
        'other':                     'Conventional / Alloys',
    }
    return family_map.get(sf, 'Conventional / Alloys')

# =============================================================================
# FIX-9: Flag low-confidence subfamily assignments
#
# conventional_Nb_based in the 169-sample training set is really a catch-all for
# simple Nb-binary/ternary compounds (alloys, borides, carbides, simple oxides
# like LiNbO2, NbOS2). The 21,263-row UCI set also contains complex layered
# Ruddlesden-Popper niobates (e.g. KCa2Nb3O10, LiRbCa2Nb3O10 -- 4+ elements,
# Nb + O, no Ti). These have no analog in the training set, so predictions for
# them are extrapolation. Flag them rather than silently trusting the vF lookup.
# =============================================================================
def is_low_confidence_extrapolation(mat_orig):
    if not isinstance(mat_orig, str):
        return False
    mat = mat_orig.replace(" ", "")

    def has(element, string):
        return re.search(f"{element}(?![a-z])", string) is not None

    sf = get_subfamily(mat_orig)
    # complex layered niobates (4+ elements, Nb + O, no Ti) -- no training analog
    if sf == 'conventional_Nb_based' and has('O', mat):
        n_elements = len(re.findall(r'[A-Z][a-z]?', mat))
        if n_elements >= 4:
            return True
    return False

# =============================================================================
# FIX-7 (unchanged) + NEW iron_245 entry -- Single source of truth for vF
# =============================================================================
VF_LOOKUP = {
    'cuprate_LSCO':              245000,
    'cuprate_YBCO':              273400,
    'cuprate_BSCCO':             304000,
    'cuprate_Tl_based':          320000,
    'cuprate_Hg_based':          290000,
    'cuprate_electron_doped':    230000,
    'cuprate_other':             273400,
    'iron_1111':                 138000,
    'iron_122':                  121500,
    'iron_111':                  109000,
    'iron_11':                    95000,
    'iron_245':                   79000,   # NEW -- Qian et al., PRL 106, 187001 (2011)
    'iron_other':                121500,
    'MgB2_type':                 560000,
    'conventional_A15':          850000,
    'conventional_Nb_based':     760000,
    'conventional_simple_metal': 1800000,
    'borocarbide':               165000,
    'chalcogenide':               95000,
    'heavy_fermion_Ce':            7595,
    'heavy_fermion_Yb':            5000,
    'heavy_fermion_115':           7595,
    'other':                     760000,
}

def get_vF(mat_orig):
    return VF_LOOKUP.get(get_subfamily(mat_orig), 760000)

AMBIGUOUS_SUBFAMILIES = {'other', 'iron_other', 'cuprate_other'}

SINGLE_VALUE_FAMILIES = {
    'Chalcogenide': 95000,
    'Mg-B related': 560000,
}

VALID_VF = np.array(sorted(set(VF_LOOKUP.values())))


def snap_to_valid(pred_ms_array):
    pred_ms_array = np.asarray(pred_ms_array).reshape(-1, 1)
    diffs = np.abs(pred_ms_array - VALID_VF)
    return VALID_VF[diffs.argmin(axis=1)]


def apply_family_override(pred_ms_array, family_array):
    pred_ms_array = np.asarray(pred_ms_array).copy()
    family_array  = np.asarray(family_array)
    for fam, val in SINGLE_VALUE_FAMILIES.items():
        pred_ms_array[family_array == fam] = val
    return pred_ms_array



In [3]:
## New version v2.0
# =============================================================================
#  External Validation — Stanev Dataset
#  Computes: family, subfamily, vF, coherence length (xi), upper critical field (Hc2)
#
#  Inputs  : stanev_recomputed.csv   ← your existing recomputed Stanev file
#                                       (must have 'material' and 'critical_temp'
#                                        columns plus the 81 Hamidieh features)
#            vF_hybrid_model.pkl     ← ML fallback model
#            vF_family_encoder.pkl   ← OneHotEncoder for family column
#
#  Outputs : stanev_with_GL_features.csv
# =============================================================================

import pandas as pd
import numpy as np
import re
import joblib
import warnings
warnings.filterwarnings('ignore')

# =============================================================================
# 1. LOAD DATA & MODELS
# =============================================================================
# ── adjust filename to match your actual Stanev CSV ──
df = pd.read_csv('stanev_top30_features_83.csv')


model   = joblib.load('vF_hybrid_model.pkl')
encoder = joblib.load('vF_family_encoder.pkl')

print(f"Stanev dataset shape : {df.shape}")
print(f"Columns sample       : {df.columns.tolist()[:5]} ...")
print(f"Tc range             : {df['critical_temp'].min():.2f} – "
      f"{df['critical_temp'].max():.2f} K\n")


import re

def get_subfamily(mat_orig):
    """
    Returns the superconductor sub-family from the ORIGINAL case formula.
    Order matters: more specific rules must come before general ones.

    CHANGE LOG (relative to fermi_velocity_169_fixed_1.ipynb):
      1. NEW: alkali-intercalated (K/Rb/Cs/Tl) FeSe-derived compounds
         (e.g. K0.8Fe1.7Se2) now route to 'iron_245' instead of being
         silently absorbed into 'iron_11' (simple binary FeSe/FeTe).
         Inserted BEFORE the iron_11 check.
      2. Ca-Pt-intercalated Fe-As "10-3-8"/"10-4-8" phase compounds
         (e.g. Ca10Pt3As8(Fe2As2)5) now route to 'iron_other' instead of
         being silently absorbed into 'iron_122' (standard ThCr2Si2-type).
         No dedicated subfamily is created, since no literature vF value
         exists for this class either way -- both 'iron_1038' and
         'iron_other' are routed through the same ML fallback path, so a
         separate label adds no behavioral difference. Inserted BEFORE
         the iron_122 check.
      3. A15 element check extended with 'Sn' AND 'In' so V3Sn- and
         V3In-type stoichiometry (V + Sn or In, no Si/Ga needed) routes
         to 'conventional_A15' instead of falling through to
         'conventional_simple_metal'.
      4. FIX: cuprate_LSCO check now excludes compounds where Ba is
         present at MAJOR stoichiometry (>=0.5) TOGETHER WITH a rare
         earth from the YBCO RE list (Er, Gd, Sm, Nd, Ho, Dy, Eu, Lu, Pr,
         Tb, Tm, Yb) -- not just Y/Bi/Tl/Hg, and not RE-presence or
         trace-Ba presence alone. Previously, a genuine YBCO-type 123
         structure such as La0.5Er0.5Ba2Cu3O was caught by the La-only
         LSCO check before ever reaching the YBCO check below it, since
         the LSCO exclusion list did not include Er. The major-Ba
         threshold (matching the real ~2-per-formula-unit Ba2Cu3 block
         that defines 123-type structures) is required in addition to RE
         presence, because compounds such as La1.6-xNd0.4SrxCuO4
         ("Nd-LSCO") are a well-established, named branch of the 214-type
         LSCO family in the literature -- Nd doping on the La site, even
         alongside trace Ba doping, does not indicate a 123-type
         structure, and an RE-alone or any-Ba exclusion would have
         wrongly pulled these genuine LSCO compounds out of cuprate_LSCO.
      5. FIX: Bi+Sr (BSCCO) check now excludes compounds that also
         contain Hg. Previously, any compound containing both Bi and Hg
         (e.g. Hg0.5Bi0.5Sr2Ca2Cu3O8.35, a genuine mixed/hybrid
         composition) was unconditionally routed to cuprate_BSCCO, since
         the BSCCO check ran before the Hg-based check with no tie-break.
         Such hybrid compositions now route to cuprate_other rather than
         being silently assigned to a single parent structure.
      6. FIX: the nested Ce+In check inside the Cu+O cuprate block (used
         to catch Ce/In-based heavy-fermion superconductors that also
         happen to satisfy has('Cu') and has('O')) now carries the same
         not has('O') guard as the standalone heavy_fermion_115 check
         above it. Previously this nested check had no O guard, so any
         genuine Ce+In co-doped cuprate (e.g. an electron-doped compound
         such as Nd1.88Ce0.12Cu0.98In0.02O4) was misrouted to
         heavy_fermion_115 instead of cuprate_electron_doped.

    This function is unchanged in every other respect, including remaining
    fully case-sensitive as originally written. It must be called on
    ORIGINAL-CASE material strings (material_orig / orig_case_map), not on
    the lowercase 'material' column -- calling it on lowercase input will
    silently fail to match any element and return 'other' for everything.
    """
    if not isinstance(mat_orig, str):
        return 'other'
    mat = mat_orig.replace(" ", "")

    def has(element, string):
        # element NOT followed by a lowercase letter -> avoids Cu matching 'ca' etc.
        return re.search(f"{element}(?![a-z])", string) is not None

    def has_major(element, string, threshold=0.5):
        m = re.search(rf"{element}(?![a-z])([0-9]*\.?[0-9]*)", string)
        if m is None:
            return False
        num = m.group(1)
        return True if num == '' else float(num) >= threshold

    def has_explicit_coefficient_ge(element, string, threshold):
        # Stricter than has_major(): requires an EXPLICIT numeric
        # coefficient >= threshold. A bare element with no written number
        # (e.g. 'Se' in 'FeSe', implicit coefficient 1) does NOT
        # automatically pass here, unlike has_major(). Needed for
        # thresholds well above 1, where "no number written" must not be
        # treated as satisfying the threshold (see CHANGE 10 below).
        m = re.search(rf"{element}(?![a-z])([0-9]+\.?[0-9]*)", string)
        if m is None:
            return False
        return float(m.group(1)) >= threshold

    def get_coeff(element, string):
        # Returns the stoichiometric coefficient of an element (1.0 if
        # present with no explicit number, 0.0 if absent). Used for
        # direct coefficient-vs-coefficient comparisons between two
        # elements (see CHANGE 13), where a simple has()/has_major()
        # presence check is not enough to determine which element is
        # structurally dominant.
        m = re.search(rf"{element}(?![a-z])([0-9]*\.?[0-9]*)", string)
        if m is None:
            return 0.0
        num = m.group(1)
        return 1.0 if num == '' else float(num)

    if has('Cu', mat) and has('O', mat):
        # Exclude Chevrel phases: Cu-Mo-S/Se compounds
        if has('Mo', mat) and any(has(x, mat) for x in ['S', 'Se']):
            return 'chalcogenide'
    # ── CUPRATES (Cu + O) ──────────────────────────────────────────────────
    # MgB2 BEFORE Fe block
    if has('Mg', mat) and has('B', mat):
        return 'MgB2_type'

    # heavy_fermion_115 BEFORE cuprate block, no-O guard
    if has('Ce', mat) and has('In', mat) and not has('O', mat):
        return 'heavy_fermion_115'

    # CUPRATES (Cu + O)
    if has('Cu', mat) and has('O', mat):
        RE = ['Er', 'Gd', 'Sm', 'Nd', 'Ho', 'Dy', 'Eu', 'Lu', 'Pr', 'Tb', 'Tm', 'Yb']
        # FIX (CHANGE 6): added not has('O', mat) guard, matching the
        # standalone heavy_fermion_115 check above. Without this guard,
        # any Ce+In co-doped cuprate (which always has O, by construction
        # of this block) was misrouted here instead of falling through to
        # the electron-doped cuprate check below.
        if has('Ce', mat) and has('In', mat) and not has('O', mat):
            return 'heavy_fermion_115'
        if has('Ce', mat) and any(has(r, mat) for r in ['Nd', 'Pr', 'Sm', 'La']):
            return 'cuprate_electron_doped'
        # FIX (CHANGE 4, updated): LSCO check now excludes compounds where
        # Ba is present at MAJOR stoichiometry (>=0.5, the real 123-type
        # Ba2Cu3-block signature) regardless of whether a second rare
        # earth is present. Literature (Ganguly & Sreedhar review of
        # RBa2Cu3O-delta phases) confirms pure LaBa2Cu3O-delta -- with La
        # as the ONLY rare earth, substituting partially onto the Ba site
        # -- is a genuine, documented member of the 123/YBCO family, not
        # a separate structure. An earlier version of this fix required a
        # SECOND rare earth (e.g. Er) alongside Ba, which correctly
        # excluded true 214-type "Nd-LSCO" compounds (La+Nd+Sr, only
        # trace Ba) but incorrectly still routed pure La+Ba2+Cu3+O7-type
        # compounds (no second RE) to cuprate_LSCO. The condition now
        # keys on major-Ba alone: trace-Ba Nd-LSCO compounds (Ba~0.14)
        # still correctly stay LSCO, since 0.14 < 0.5, while major-Ba
        # La-123 compounds (Ba~1.3-2) now correctly route to cuprate_YBCO.
        if has('La', mat) and not any(has(x, mat) for x in ['Y', 'Bi', 'Tl', 'Hg']) and not has_major('Ba', mat):
            return 'cuprate_LSCO'
        # FIX (CHANGE 5, revised) + FIX (CHANGE 13): BSCCO check now
        # requires Bi's coefficient to strictly exceed both Hg's and Tl's
        # coefficients, rather than excluding on bare presence of either
        # element. Literature confirms both Bi2-xHgxSr2-yBayCaCu2O8-type
        # compounds (Toulemonde and Odier, arXiv:cond-mat/0304373; Jabur
        # et al., Eng. Technol. J. 31, 2650, 2013) and Tl1-xBix-1223/1212/
        # 1201-type compounds (Hur et al., Physica C 253, 109, 1995; Lee
        # et al., J. Appl. Phys. 80, 326, 1996) are treated in the
        # primary literature as substitutional variants of their
        # respective PARENT structures (Bi-2212 for the Hg case, Tl-1223/
        # 1212/1201 for the Tl case), with the parent's defining element
        # remaining structurally dominant at substitution levels up to
        # ~50%. STRICT inequality (not >=) is used deliberately: at an
        # exact 1:1 Hg:Bi ratio the compound is treated as Hg-based (its
        # own established convention elsewhere in this function), while
        # at an exact 1:1 Tl:Bi ratio the compound is treated as Tl-based
        # (e.g. "Tl-1201", "Tl-1212" oxycarbonate phases named as such in
        # the primary literature regardless of the 1:1 substitution
        # level) -- so Bi must strictly exceed each competing element's
        # coefficient, not merely equal it, for BSCCO to apply.
        if has('Bi', mat) and has('Sr', mat) and get_coeff('Bi', mat) > get_coeff('Hg', mat) and get_coeff('Bi', mat) > get_coeff('Tl', mat):
            return 'cuprate_BSCCO'
        # FIX (CHANGE 8 continued): YBCO check's rare-earth trigger now
        # also includes La. Previously only Y and the RE list (Er, Gd,
        # Sm, Nd, Ho, Dy, Eu, Lu, Pr, Tb, Tm, Yb) reached this check --
        # La was excluded here too (it is handled as its own special
        # case throughout this function). Combined with the LSCO-check
        # fix above, a pure La+Ba(major)+Cu3+O7-type compound (e.g.
        # La1.5Ba1.3Ca0.2Cu3O6.897, no second rare earth) would otherwise
        # fall through BOTH the LSCO check (correctly excluded) and this
        # YBCO check (La not recognized as a trigger), landing in
        # cuprate_other instead of the correct cuprate_YBCO.
        if has('Ba', mat) and (has('Y', mat) or has('La', mat) or any(has(r, mat) for r in RE)):
            return 'cuprate_YBCO'
        # Tl before Hg
        if has('Tl', mat):
            return 'cuprate_Tl_based'
        if has('Hg', mat):
            return 'cuprate_Hg_based'
        return 'cuprate_other'


    # ── IRON-BASED (Fe) ───────────────────────────────────────────────────
    if has_major('Fe', mat):
        # NEW (CHANGE 2): Ca-Pt-arsenide "10-3-8"/"10-4-8" phase.
        # Requires Ca AND Pt together (the defining Ca10(Pt3As8)(Fe2As2)5
        # signature). Routed to iron_other (same ML-fallback path as
        # iron_1038 would have used) rather than a separate named
        # subfamily, since no literature vF value exists for this class.
        # Must come BEFORE the iron_1111 and iron_122 checks.
        if has('Ca', mat) and has('Pt', mat) and has('As', mat):
            return 'iron_other'

        # NEW (CHANGE 11): thick perovskite-blocking-layer families
        # (21113 e.g. Sr2VFeAsO3; 4226 e.g. Sr4V2Fe2As2O6, Ca4Al2Fe2As2O6;
        # 3225, 5229, 62212 and further homologues with n=3,4,5,6 perovskite
        # layers; 22241 e.g. Ba2Ti2Fe2As4O). Must come BEFORE the iron_1111
        # check, since all of these contain As+O and would otherwise be
        # caught there. Literature (Ogino et al., Appl. Phys. Lett. 97,
        # 072506, 2010; Nakamura & Machida, arXiv:1004.4741; Kotegawa et
        # al., arXiv:1009.5491; and the JACS report on Ba2Ti2Fe2As4O)
        # confirms these are a structurally distinct family from thin-layer
        # 1111 (LaFeAsO-type): the defining difference among all iron-based
        # numbering families is the composition and thickness of the
        # non-superconducting blocking layer between FeAs sheets, and this
        # family's blocking layer is a genuine multi-layer perovskite block
        # containing a SECOND transition/post-transition metal (V, Ti, Sc,
        # Al, Mg, or Cr) beyond Fe itself -- absent from true 1111, whose
        # blocking layer contains only the rare-earth/alkaline-earth cation
        # and O/H/F. No literature vF exists for this niche, structurally
        # complex family -- routed to iron_other (ML fallback).
        BLOCKING_LAYER_METALS = ['V', 'Ti', 'Sc', 'Al', 'Mg', 'Cr']
        if has('As', mat) and has('O', mat) and any(has(m, mat) for m in BLOCKING_LAYER_METALS):
            return 'iron_other'

        # NEW (CHANGE 12): 42210/42214 family (Ln4Fe2As2TeO4-type, e.g.
        # Pr4Fe2As2Te0.88O4, Sm4Fe2As2Te0.72F1.2O2.8). Must come BEFORE the
        # iron_1111 check for the same reason as CHANGE 11 (these contain
        # As+O). Literature (the 42214 structure report, arXiv:1311.5437)
        # confirms this family is isoconfigurational with the unrelated
        # La3Cu4P4O2/La3Ni4P4O2 ("3442") structure type -- built from
        # fluorite-type Ln2O2 layers and anti-fluorite-type Fe2As2 layers
        # stacked in a sequence and layer count genuinely different from
        # 1111's simple ZrCuSiAs stacking, and further distinguished by the
        # blocking layer's tellurium content. No literature vF exists --
        # routed to iron_other (ML fallback).
        if has('As', mat) and has('O', mat) and has('Te', mat):
            return 'iron_other'

        # FIX (CHANGE 8, extended): 'or has('F', mat)' added alongside H.
        # Literature (Matsuishi et al., J. Am. Chem. Soc. 130, 14428, 2008;
        # multiple independent AeFeAsF band-structure and synthesis papers)
        # confirms fluoroarsenide "1111" compounds (CaFeAsF, SrFeAsF, and
        # rare-earth-codoped variants such as Sr0.6La0.4FeAsF) adopt the
        # same ZrCuSiAs-type structure as the oxide 1111 family, with F-
        # substituting directly for O2- in the blocking layer -- the
        # isovalent counterpart to the H- substitution already handled
        # above. Without this, compounds like Ca1Fe0.9Co0.1As1F1 (no O or H)
        # failed this check and fell through to iron_122 instead, since Ca
        # then triggered that branch's alkaline-earth condition. The two
        # thick-blocking-layer checks above (CHANGE 11, CHANGE 12) run
        # first so that genuine thin-layer 1111 compounds (no second
        # transition metal, no Te) are the only ones reaching this check.
        if has('As', mat) and (has('O', mat) or has('H', mat) or has('F', mat)):
            return 'iron_1111'

        # NEW (CHANGE 10): quasi-1D "ladder" AFe2Se3 phase (A = Ba, K, Cs).
        # Must come BEFORE both the iron_245 and iron_11 checks below, since
        # these compounds would otherwise be caught by either (they contain
        # the same alkali/alkaline-earth intercalants as iron_245, and no
        # As, same as iron_11). Literature (Krzton-Maziopa et al., J. Phys.
        # Condens. Matter 23, 402201, 2011; Caron et al., PRB 84, 180409,
        # 2011; Nature Materials 14, 745, 2015) confirms AFe2Se3 is a
        # structurally DISTINCT family from both: built from quasi-1D
        # two-leg Fe-Se ladders (edge-sharing FeSe4 tetrahedra), not the 2D
        # FeSe layers found in either iron_11 (simple binary FeSe/FeTe) or
        # iron_245 (2D AFe2Se2 intercalated layers). Detected here by
        # Fe:Se stoichiometry near 2:3 (vs 1:1 for iron_11, 1:2 for
        # iron_245). No literature vF exists for this niche, primarily
        # magnetically-studied family -- routed to iron_other (ML fallback)
        # rather than a hand-picked or borrowed lookup value.
        if has_explicit_coefficient_ge('Se', mat, 2.5) and not has('As', mat):
            return 'iron_other'

        # NEW (CHANGE 1, extended): alkali- OR alkaline-earth-intercalated
        # 245-type selenide. Ca/Sr/Ba/Eu/Yb added alongside the original
        # K/Rb/Cs/Tl/Na alkali list -- literature confirms AFe2Se2 (A = Ca,
        # Sr, Ba, Eu, Yb, in addition to the alkalis) shares the same
        # ThCr2Si2-type "122-like" intercalated structure, distinct from
        # simple binary FeSe/FeTe (iron_11). Detected by Fe:Se ~= 1:2
        # stoichiometry (the ladder-compound check above, which requires
        # Se >= 2.5, already excludes the 2:3-stoichiometry AFe2Se3 ladder
        # compounds from reaching this branch).
        if any(has(a, mat) for a in ['Li', 'K', 'Rb', 'Cs', 'Tl', 'Na', 'Ca', 'Sr', 'Ba', 'Eu', 'Yb']) and has('Se', mat) and not has('As', mat):
            return 'iron_245'

        if (has('Te', mat) or has('Se', mat)) and not has('As', mat):
            return 'iron_11'
        # 122 before 111
        if has('As', mat) and any(has(x, mat) for x in ['Ba','Sr','Ca','K','Eu']):
            return 'iron_122'
        if has('As', mat) and (has('Li', mat) or has('Na', mat)):
            return 'iron_111'
        if has('As', mat) or has('P', mat):
            return 'iron_other'
        return 'iron_other'

    # ── HEAVY FERMION ─────────────────────────────────────────────────────
    if has('Ce', mat) and any(has(x, mat) for x in ['Pt', 'Pd', 'Rh', 'Co', 'In', 'Ru', 'Cu', 'Si']):
        return 'heavy_fermion_Ce'
    if has('Yb', mat) and any(has(x, mat) for x in ['Rh', 'Pt', 'Ni']):
        return 'heavy_fermion_Yb'

    # ── BOROCARBIDE  (must come BEFORE generic C check) ───────────────────
    RE2 = ['Lu', 'Ho', 'Er', 'Dy', 'Tm', 'Gd', 'Y', 'Tb']
    if has('Ni', mat) and has('B', mat) and has('C', mat) and any(has(r, mat) for r in RE2):
        return 'borocarbide'

    # ── CONVENTIONAL A15 ──────────────────────────────────────────────────
    if has('Nb', mat) and any(has(x, mat) for x in ['Sn', 'Al', 'Ge']) and not has_major('Fe', mat):
        return 'conventional_A15'
    # MODIFIED (CHANGE 3): added 'Sn', 'In', and 'Al'. V-Al alloys are
    # explicitly documented A15-structure superconductors (see e.g.
    # Zeitschrift fur Physik B, "Critical temperature ... of superconducting
    # V-Ga-Al alloys with A15 structure").
    if has('V', mat) and any(has(x, mat) for x in ['Si', 'Ga', 'Sn', 'In', 'Al']) and not has_major('Fe', mat):
        return 'conventional_A15'

    # ── CONVENTIONAL Nb-BASED (RENAMED, CHANGE 9) ───────────────────────────
    # Was 'conventional_NbTi'. Renamed to 'conventional_Nb_based' since the
    # actual contents of this bucket are general Nb binary/ternary
    # compounds (NbB, NbC, Nb-Cr, Nb-Ga, Nb-Hf, Nb-Ir, Li0.5NbO2, NbOS2),
    # not specifically Nb-Ti alloys -- the old name implied a narrower,
    # inaccurate structural claim.
    if has('Nb', mat):
        return 'conventional_Nb_based'

    # ── CHALCOGENIDE (non-Fe) ─────────────────────────────────────────────
    if any(has(x, mat) for x in ['Se', 'Te', 'S']) and not has_major('Fe', mat):
        return 'chalcogenide'

    # ── MgB2-TYPE ─────────────────────────────────────────────────────────
    if has('Mg', mat) and has('B', mat):
        return 'MgB2_type'

    # ── BISMUTHATE (NEW, CHANGE 7) ──────────────────────────────────────────
    # BaBiO3-derived perovskite/CDW superconductors: BaPb1-xBixO3 (BPBO) and
    # Ba1-xKxBiO3 (BKBO). Structurally and mechanistically distinct from the
    # sp-band alloys in conventional_simple_metal (non-magnetic CDW-derived
    # perovskite oxide, not a simple metallic alloy) -- see e.g. Nature
    # Materials/Nature Communications reviews of the BPBO/BKBO family.
    # Must come BEFORE the simple-metals check, since Pb/Al/In there would
    # otherwise catch these compounds first.
    # No literature vF value is assigned for this class -- routed to
    # 'other' (ML fallback), matching how iron_1038 and other
    # structurally-distinct-but-unvalued classes are handled.
    if has('Ba', mat) and has('Bi', mat) and has('O', mat) and any(has(x, mat) for x in ['Pb', 'K']):
        return 'other'

    # ── SIMPLE METALS ─────────────────────────────────────────────────────
    if (any(has(x, mat) for x in ['Pb', 'Al', 'Sn', 'In'])
            and not has_major('Fe', mat)
            and not has('Cu', mat)):
        return 'conventional_simple_metal'

    return 'other'


def get_family(mat_orig):
    sf = get_subfamily(mat_orig)
    family_map = {
        'cuprate_LSCO':              'Cuprates',
        'cuprate_YBCO':              'Cuprates',
        'cuprate_BSCCO':             'Cuprates',
        'cuprate_Tl_based':          'Cuprates',
        'cuprate_Hg_based':          'Cuprates',
        'cuprate_electron_doped':    'Cuprates',
        'cuprate_other':             'Cuprates',
        'iron_1111':                 'Iron-based',
        'iron_122':                  'Iron-based',
        'iron_111':                  'Iron-based',
        'iron_11':                   'Iron-based',
        'iron_245':                  'Iron-based',   # NEW
        'iron_other':                'Iron-based',
        'MgB2_type':                 'Mg-B related',
        'conventional_A15':          'Conventional / Alloys',
        'conventional_Nb_based':     'Conventional / Alloys',
        'conventional_simple_metal': 'Conventional / Alloys',
        'borocarbide':               'Conventional / Alloys',
        'chalcogenide':              'Chalcogenide',
        'heavy_fermion_Ce':          'Heavy Fermion',
        'heavy_fermion_Yb':          'Heavy Fermion',
        'heavy_fermion_115':         'Heavy Fermion',
        'other':                     'Conventional / Alloys',
    }
    return family_map.get(sf, 'Conventional / Alloys')


# =============================================================================
# vF LOOKUP TABLE -- direct-lookup subfamilies only.
#
# IMPORTANT: 'other', 'iron_other', and 'cuprate_other' are INTENTIONALLY
# ABSENT from this dictionary. These are the catch-all buckets for
# structurally ambiguous or genuinely mixed compounds (e.g. the Ca-Pt-As
# "10-3-8" phase inside iron_other, Hg+Bi hybrid cuprates inside
# cuprate_other, and chemically unrelated compounds like bismuthate
# perovskites or Fe-Ni-Zr Laves phases inside the generic 'other'). No
# single literature-backed Fermi velocity exists for these buckets, since
# they are deliberately heterogeneous by construction, not a real physical
# family. Per the actual training pipeline (vF_xi_Hc2.ipynb, Section 2:
# AMBIGUOUS_SUBFAMILIES / is_ambiguous_row), rows in these three subfamilies
# are ALWAYS routed to the trained ML fallback model (final_model.predict,
# followed by snap_to_valid) instead of a fixed lookup value. An earlier
# version of this file incorrectly gave these three subfamilies a borrowed
# or arbitrary constant (cuprate_other = cuprate_YBCO's value, iron_other =
# iron_122's value, other = an unexplained 760,000 default) -- these were
# never real citations and did not match how the actual pipeline behaves,
# so get_vF() below now raises rather than silently returning a number that
# looks citation-backed but isn't.
#
# conventional_A15 updated to the literature value: Mentink et al.,
# Supercond. Sci. Technol. 30, 025006 (2017) -- ab initio DFT calculation
# of Nb3Sn, RMS Fermi velocity vF = 2.121e5 m/s, cross-validated against
# four independent experimental sources (Lim, Schachinger, Orlando, Ghosh)
# in the same paper. Replaces the previous unsourced 850,000 m/s.
#
# conventional_Nb_based updated to: Prozorov, Zarea, and Sauls, arXiv:
# 2207.11829 (2022), citing the underlying DFT calculation of elemental Nb
# by Zarea, Ueki, and Sauls, arXiv:2201.07403 (2022). Two-band-averaged
# RMS Fermi velocity v = 7.26e5 m/s. This is elemental niobium's own value,
# used as the representative constant for the whole conventional_Nb_based
# bucket (NbB, NbC, Nb-Cr, Nb-Ga, Nb-Hf, Nb-Ir, Li0.5NbO2, NbOS2, etc.)
# since Nb is the one element common to every compound in this bucket;
# compound-specific corrections (carbide vs boride vs oxide bonding) have
# no citable literature basis and are not applied.
# =============================================================================
VF_LOOKUP = {
    'cuprate_LSCO':              245000,
    'cuprate_YBCO':              273400,
    'cuprate_BSCCO':             304000,
    'cuprate_Tl_based':          320000,
    'cuprate_Hg_based':          290000,
    'cuprate_electron_doped':    230000,
    'iron_1111':                 152000,
    'iron_122':                  91981,
    'iron_111':                  45600,
    'iron_11':                    13700,
    'iron_245':                   79000,   # Qian et al., PRL 106, 187001 (2011)
    'MgB2_type':                 560000,
    'conventional_A15':          212100,   # Mentink et al., SST 30, 025006 (2017)
    'conventional_Nb_based':     726000,   # Prozorov, Zarea & Sauls, arXiv:2207.11829 (2022)
    'conventional_simple_metal': 1800000,
    'borocarbide':               276000,
    'chalcogenide':               95000,
    'heavy_fermion_Ce':            7595,
    'heavy_fermion_Yb':            5000,
    'heavy_fermion_115':           7595,
    # 'cuprate_other', 'iron_other', 'other' deliberately absent -- see above.
}

# def is_low_confidence_extrapolation(mat_orig):
#     if not isinstance(mat_orig, str):
#         return False
#     mat = mat_orig.replace(" ", "")

#     def has(element, string):
#         return re.search(f"{element}(?![a-z])", string) is not None

#     sf = get_subfamily(mat_orig)
#     # complex layered niobates (4+ elements, Nb + O, no Ti) -- no training analog
#     if sf == 'conventional_Nb_based' and has('O', mat):
#         n_elements = len(re.findall(r'[A-Z][a-z]?', mat))
#         if n_elements >= 4:
#             return True
#     return False

# def get_vF(mat_orig):
#     return VF_LOOKUP.get(get_subfamily(mat_orig), 760000)

AMBIGUOUS_SUBFAMILIES = {'other', 'iron_other', 'cuprate_other'}

SINGLE_VALUE_FAMILIES = {
    'Chalcogenide': 95000,
    'Mg-B related': 560000,
}

VALID_VF = np.array(sorted(set(VF_LOOKUP.values())))


def snap_to_valid(pred_ms_array):
    pred_ms_array = np.asarray(pred_ms_array).reshape(-1, 1)
    diffs = np.abs(pred_ms_array - VALID_VF)
    return VALID_VF[diffs.argmin(axis=1)]


def apply_family_override(pred_ms_array, family_array):
    pred_ms_array = np.asarray(pred_ms_array).copy()
    family_array  = np.asarray(family_array)
    for fam, val in SINGLE_VALUE_FAMILIES.items():
        pred_ms_array[family_array == fam] = val
    return pred_ms_array

Stanev dataset shape : (250, 82)
Columns sample       : ['material', 'critical_temp', 'mean_atomic_mass', 'wtd_mean_atomic_mass', 'gmean_atomic_mass'] ...
Tc range             : 0.00 – 150.00 K



In [4]:

# =============================================================================
# 4. ASSIGN FAMILY & SUBFAMILY
# =============================================================================
# material column — adjust name if yours is different (e.g. 'formula', 'name')
MATERIAL_COL = 'material'

df['subfamily'] = df[MATERIAL_COL].apply(get_subfamily)
df['family']    = df[MATERIAL_COL].apply(get_family)
df['is_ambiguous'] = df['subfamily'].isin(AMBIGUOUS_SUBFAMILIES)

print("Family distribution (Stanev):")
print(df['family'].value_counts().to_string())
print(f"\nSubfamily distribution:")
print(df['subfamily'].value_counts().to_string())
print(f"\nAmbiguous rows : {df['is_ambiguous'].sum()} "
      f"({df['is_ambiguous'].mean()*100:.1f}%)")

# =============================================================================
# 5. PREDICT vF — hybrid: direct lookup + ML fallback
# =============================================================================
NUMERIC_FEATURES = [

"range_atomic_mass",
"wtd_gmean_Density",
"range_Valence",
"wtd_mean_FusionHeat",
"wtd_mean_ThermalConductivity",
"mean_ThermalConductivity"
"wtd_std_Valence",
"wtd_mean_Valence",
"wtd_std_ThermalConductivity",
"wtd_std_ElectronAffinity",
"wtd_entropy_ElectronAffinity"

]

# check all required numeric features are present
missing = [f for f in NUMERIC_FEATURES if f not in df.columns]
if missing:
    print(f"\nWARNING: Missing features for ML fallback: {missing}")
    print("These must be present in your recomputed Stanev CSV.")
    print("ML fallback rows will default to subfamily lookup value.\n")

vf_pred = np.zeros(len(df))
pred_source = np.array(['lookup'] * len(df), dtype=object)

# --- non-ambiguous: direct lookup ---
lookup_mask = ~df['is_ambiguous'].values
vf_pred[lookup_mask] = df.loc[lookup_mask, 'subfamily'].map(VF_LOOKUP).values

##
# Run this diagnostic BEFORE your existing Cell 5 vF-prediction block,
# right after df['is_ambiguous'] is computed, to pinpoint the exact cause.

print("=== DIAGNOSTIC ===")
print(f"Total rows: {len(df)}")
print(f"Ambiguous rows (is_ambiguous=True): {df['is_ambiguous'].sum()}")
print(f"Non-ambiguous rows: {(~df['is_ambiguous']).sum()}")

NUMERIC_FEATURES = [
    "mean_ThermalConductivity", "wtd_mean_Valence", "range_atomic_mass",
    "wtd_std_ThermalConductivity", "wtd_gmean_Density", "wtd_mean_FusionHeat",
    "wtd_std_FusionHeat", "wtd_std_Valence", "range_Valence",
    "mean_ElectronAffinity", "wtd_mean_ThermalConductivity"
]
missing = [f for f in NUMERIC_FEATURES if f not in df.columns]
print(f"Missing NUMERIC_FEATURES columns: {missing}")

if not missing:
    # check for NaN within the ambiguous rows specifically, even if columns exist
    amb = df[df['is_ambiguous']]
    nan_counts = amb[NUMERIC_FEATURES].isna().sum()
    print("NaN counts in NUMERIC_FEATURES, ambiguous rows only:")
    print(nan_counts[nan_counts > 0].to_string() if nan_counts.sum() > 0 else "  (none)")

# Check for any subfamily string that doesn't map cleanly (whitespace, case, etc.)
non_amb = df[~df['is_ambiguous']]
mapped = non_amb['subfamily'].map(VF_LOOKUP)
unmapped_mask = mapped.isna()
if unmapped_mask.sum() > 0:
    print(f"\n!!! {unmapped_mask.sum()} NON-ambiguous rows have a subfamily NOT in VF_LOOKUP:")
    print(non_amb.loc[unmapped_mask, 'subfamily'].value_counts().to_string())
else:
    print("\nAll non-ambiguous rows map cleanly to a VF_LOOKUP value.")

##

# --- ambiguous: ML fallback ---
amb_mask = df['is_ambiguous'].values
if amb_mask.sum() > 0 and len(missing) == 0:
    fam_enc  = encoder.transform(df.loc[amb_mask, ['family']])
    X_num    = df.loc[amb_mask, NUMERIC_FEATURES].values
    X_amb    = np.hstack([X_num, fam_enc])
    raw_log  = model.predict(X_amb)
    raw_ms   = 10 ** raw_log
    snapped  = snap_to_valid(raw_ms)
    snapped  = apply_family_override(snapped, df.loc[amb_mask, 'family'].values)
    vf_pred[amb_mask]     = snapped
    pred_source[amb_mask] = 'model'
# elif amb_mask.sum() > 0 and len(missing) > 0:
#     # fallback to 'other' lookup value when features missing
#     vf_pred[amb_mask] = AMBIGUOUS_SUBFAMILIES['other']
#     pred_source[amb_mask] = 'lookup_default'

df['vF_predicted_ms']   = vf_pred.astype(int)
df['prediction_source'] = pred_source

print(f"\nvF prediction source breakdown:")
print(df['prediction_source'].value_counts().to_string())
print(f"\nvF range: {df['vF_predicted_ms'].min():,} – "
      f"{df['vF_predicted_ms'].max():,} m/s")


Family distribution (Stanev):
family
Conventional / Alloys    119
Cuprates                  71
Iron-based                30
Chalcogenide              19
Mg-B related               6
Heavy Fermion              5

Subfamily distribution:
subfamily
conventional_simple_metal    55
conventional_Nb_based        45
cuprate_YBCO                 29
chalcogenide                 19
cuprate_LSCO                 15
conventional_A15             14
iron_122                      9
cuprate_electron_doped        8
iron_245                      8
cuprate_BSCCO                 7
cuprate_Tl_based              6
MgB2_type                     6
iron_1111                     6
cuprate_Hg_based              6
borocarbide                   4
iron_11                       4
iron_111                      3
heavy_fermion_Ce              3
other                         1
heavy_fermion_Yb              1
heavy_fermion_115             1

Ambiguous rows : 1 (0.4%)

These must be present in your recomputed Stanev CSV.
M

In [ ]:
# =============================================================================
# REPLACEMENT FOR CELL 6 ("6. COMPUTE GL FEATURES") IN
# BCS_GL_External_validation.ipynb
#
# WHY THIS REPLACES THE OLD CELL 6
# ---------------------------------
# The old cell computed xi and Hc2 for the Stanev compounds using their
# REAL, measured critical_temp:
#
#     Tc = np.clip(df['critical_temp'].values, 0.01, None)   # <- leakage
#     energy_gap = 1.764 * Kb * Tc
#     df['coherence_length'] = (h_cut * vF) / (Pi * energy_gap)
#
# xi and Hc2 are then fed into the trained stacking model (cell 9) as two
# of its 30 input features, and the model is scored against that SAME
# critical_temp. Since xi ~ vF/Tc and Hc2 ~ Tc^2/vF^2 by construction, the
# model is partially given the answer as an input feature. This inflates
# the external-validation R^2/RMSE/MAE and is inconsistent with how ξ/Hc2
# were computed for the UCI training set, where the leakage-free
# out-of-fold (OOF) Tc from Section 2.3 was used instead of real Tc.
#
# THE FIX
# -------
# Train a Tc-proxy model on the full UCI training set (all 21,263 rows),
# using the SAME 81-feature + family/subfamily input spec and the SAME
# XGBoost hyperparameters as the OOF-Tc model in vF_xi_Hc2.ipynb. Note this
# is NOT the same object as the OOF model there -- cross_val_predict()
# doesn't preserve a single fitted model, it only returns fold-wise
# predictions on UCI's own rows, which is unusable for a fully external
# dataset like Stanev. Refitting on all of UCI is the correct move here:
# Stanev compounds are disjoint from UCI, so there is no leakage in fitting
# on 100% of UCI and predicting on Stanev (this is exactly what OOF avoided
# needing to do for UCI's own rows, where fold-holdout was necessary
# specifically because the same rows were used for both fitting and
# feature construction).
#
# IMPORTANT CAVEAT -- READ BEFORE RUNNING
# -----------------------------------------
# Your existing recomputed Stanev file (stanev_top30_features_83.csv /
# stanev_250_new.csv) only contains 27 of the 81 Hamidieh compositional
# features -- specifically the subset in TOP30_COMPOSITIONAL_SPEC, because
# it was built to match the FINAL model's 30-feature input, not the
# OOF-Tc model's 81-feature input. The OOF-Tc model in vF_xi_Hc2.ipynb
# was trained on all 81 features. Retraining that exact same model is not
# reproducible on Stanev with only 27 features available.
#
# This script trains the Tc-proxy model on the 27 compositional features
# actually available for Stanev (plus family/subfamily), NOT the full 81.
# This is a legitimate, honestly-reported choice -- fitting the proxy
# model on whatever feature subset is available for both UCI and Stanev
# is still leakage-free w.r.t. Tc -- but it is a materially different
# model from the original 81-feature OOF-Tc model, so the resulting Tc
# proxy for Stanev compounds will not numerically match what an 81-feature
# version would have produced. If you want the closer match, first
# recompute the FULL 81 Hamidieh features for the 250 Stanev compounds
# (using the compute_composition_features() machinery already in cell 0
# of BCS_GL_External_validation.ipynb, extended to all 81 properties
# rather than just the 27-feature TOP30_COMPOSITIONAL_SPEC subset), then
# rerun this script with FEATURES_FOR_PROXY = the full 81-name list.
#
# OUTPUT
# ------
# df['Tc_proxy']            -- leakage-free Tc estimate for each Stanev row
# df['coherence_length']    -- recomputed from Tc_proxy, not real Tc
# df['upper_critical_field']-- recomputed from Tc_proxy, not real Tc
#
# Downstream (old cell 9) should be re-run unchanged after this -- it
# will pick up the corrected coherence_length / upper_critical_field
# columns automatically via top30_cols.
# =============================================================================

import pandas as pd
import numpy as np
from sklearn.preprocessing import OrdinalEncoder
from xgboost import XGBRegressor

# -----------------------------------------------------------------------
# 1. Load UCI training data (the same file vF_xi_Hc2.ipynb's OOF-Tc step
#    reads from) and the Stanev dataframe (`df`, already carrying
#    vF_predicted_ms, family, subfamily, and the 27-feature TOP30 subset
#    from earlier cells in this notebook).
# -----------------------------------------------------------------------
uci = pd.read_csv('UCI_family_subfamily_vF.csv')

# The 27 compositional features actually available for Stanev (see
# CAVEAT above -- swap in the full 81-name list here if you recompute them)
FEATURES_FOR_PROXY = [
# 'range_ThermalConductivity',         
# 'wtd_mean_Valence',                   
# 'wtd_std_ThermalConductivity',    
# 'wtd_gmean_Valence',              
# 'std_atomic_mass',                
# 'range_atomic_radius',            
# 'wtd_gmean_ThermalConductivity',  
# 'wtd_entropy_ThermalConductivity',
# 'gmean_Density',                  
# 'wtd_std_ElectronAffinity',       
# 'wtd_mean_Density',               
# 'wtd_std_Valence',                
# 'wtd_mean_ThermalConductivity',   
# 'mean_Density',                   
# 'wtd_std_atomic_radius',          
# 'wtd_entropy_FusionHeat',         
# 'wtd_entropy_Density',            
# 'wtd_entropy_atomic_mass',        
# 'wtd_std_FusionHeat',             
# 'wtd_gmean_ElectronAffinity',     
# 'std_ThermalConductivity',        
# 'wtd_gmean_atomic_mass',          
# 'wtd_mean_ElectronAffinity',      
# 'std_atomic_radius',              
# 'wtd_std_Density',                
# 'wtd_std_atomic_mass',            
# 'std_ElectronAffinity',      
'mean_atomic_mass', 'wtd_mean_atomic_mass',
       'gmean_atomic_mass', 'wtd_gmean_atomic_mass', 'entropy_atomic_mass',
       'wtd_entropy_atomic_mass', 'range_atomic_mass', 'wtd_range_atomic_mass',
       'std_atomic_mass', 'wtd_std_atomic_mass', 'mean_fie', 'wtd_mean_fie',
       'gmean_fie', 'wtd_gmean_fie', 'entropy_fie', 'wtd_entropy_fie',
       'range_fie', 'wtd_range_fie', 'std_fie', 'wtd_std_fie',
       'mean_atomic_radius', 'wtd_mean_atomic_radius', 'gmean_atomic_radius',
       'wtd_gmean_atomic_radius', 'entropy_atomic_radius',
       'wtd_entropy_atomic_radius', 'range_atomic_radius',
       'wtd_range_atomic_radius', 'std_atomic_radius', 'wtd_std_atomic_radius',
       'mean_Density', 'wtd_mean_Density', 'gmean_Density',
       'wtd_gmean_Density', 'entropy_Density', 'wtd_entropy_Density',
       'range_Density', 'wtd_range_Density', 'std_Density', 'wtd_std_Density',
       'mean_ElectronAffinity', 'wtd_mean_ElectronAffinity',
       'gmean_ElectronAffinity', 'wtd_gmean_ElectronAffinity',
       'entropy_ElectronAffinity', 'wtd_entropy_ElectronAffinity',
       'range_ElectronAffinity', 'wtd_range_ElectronAffinity',
       'std_ElectronAffinity', 'wtd_std_ElectronAffinity', 'mean_FusionHeat',
       'wtd_mean_FusionHeat', 'gmean_FusionHeat', 'wtd_gmean_FusionHeat',
       'entropy_FusionHeat', 'wtd_entropy_FusionHeat', 'range_FusionHeat',
       'wtd_range_FusionHeat', 'std_FusionHeat', 'wtd_std_FusionHeat',
       'mean_ThermalConductivity', 'wtd_mean_ThermalConductivity',
       'gmean_ThermalConductivity', 'wtd_gmean_ThermalConductivity',
       'entropy_ThermalConductivity', 'wtd_entropy_ThermalConductivity',
       'range_ThermalConductivity', 'wtd_range_ThermalConductivity',
       'std_ThermalConductivity', 'wtd_std_ThermalConductivity',
       'mean_Valence', 'wtd_mean_Valence', 'gmean_Valence',
       'wtd_gmean_Valence', 'entropy_Valence', 'wtd_entropy_Valence',
       'range_Valence', 'wtd_range_Valence', 'std_Valence', 'wtd_std_Valence',     
]
# Keep only columns that actually exist in both frames (defensive)
FEATURES_FOR_PROXY = [c for c in FEATURES_FOR_PROXY if c in uci.columns and c in df.columns]
print(f"Tc-proxy compositional features used: {len(FEATURES_FOR_PROXY)}")

missing_in_uci = [c for c in FEATURES_FOR_PROXY if c not in uci.columns]
missing_in_stanev = [c for c in FEATURES_FOR_PROXY if c not in df.columns]
if missing_in_uci or missing_in_stanev:
    print(f"WARNING -- missing in UCI: {missing_in_uci}")
    print(f"WARNING -- missing in Stanev df: {missing_in_stanev}")

# -----------------------------------------------------------------------
# 2. Encode family + subfamily the same way vF_xi_Hc2.ipynb did for the
#    OOF-Tc model (OrdinalEncoder, unseen -> -1). Fit ONLY on UCI, then
#    transform both UCI and Stanev with that same fitted encoder, so a
#    Stanev subfamily label UCI never saw maps safely to -1 rather than
#    crashing or silently reusing a UCI category by coincidence.
# -----------------------------------------------------------------------
# enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
# uci_cat = enc.fit_transform(uci[['family', 'subfamily']])
# stanev_cat = enc.transform(df[['family', 'subfamily']])

X_uci= uci[FEATURES_FOR_PROXY].values
# X_uci = np.hstack([X_uci_num, uci_cat])
y_uci = uci['critical_temp'].values

# df = pd.read_csv('stanev_top30_features_83.csv')
X_stanev = df[FEATURES_FOR_PROXY].values
# X_stanev = np.hstack([X_stanev_num, stanev_cat])

# -----------------------------------------------------------------------
# 3. Fit the Tc-proxy model on ALL of UCI (no held-out fold needed --
#    Stanev compounds are already confirmed disjoint from UCI, so fitting
#    on the full UCI set and predicting on Stanev introduces no leakage).
#    Same hyperparameters as vF_xi_Hc2.ipynb's OOF-Tc model.
# -----------------------------------------------------------------------
tc_proxy_model = XGBRegressor(
    n_estimators=800,
    learning_rate=0.03,
    gamma=0.05,
    min_child_weight=9,
    max_depth=4,
    subsample=1,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=0,
    n_jobs=-1,
)
tc_proxy_model.fit(X_uci, y_uci)

from sklearn.metrics import r2_score, mean_squared_error
# -----------------------------------------------------------------------
# 4. Predict Tc_proxy for the 250 Stanev compounds. This is genuinely
#    leakage-free: the model has never seen Stanev's features or Tc.
# -----------------------------------------------------------------------
tc_proxy_stanev = tc_proxy_model.predict(X_stanev)
tc_proxy_stanev = np.clip(tc_proxy_stanev, 0.01, None)
df['Tc_proxy'] = tc_proxy_stanev

# Report how far the proxy is from real Tc (informational only -- do NOT
# use this to "correct" the proxy, that would reintroduce leakage. This
# number belongs in your methods/limitations text, not in the pipeline.)
proxy_mae = np.mean(np.abs(df['Tc_proxy'].values - df['critical_temp'].values))
proxy_r2 = r2_score(df['critical_temp'].values, df['Tc_proxy'].values)
print(f"\nTc_proxy vs real Tc on Stanev (informational, NOT leakage-safe "
      f"if used for anything beyond reporting):")
print(f"  R^2 : {proxy_r2:.4f}")
print(f"  MAE : {proxy_mae:.4f} K")

# -----------------------------------------------------------------------
# 5. Recompute xi and Hc2 using Tc_proxy instead of real critical_temp.
#    This is the actual fix -- everything above exists to produce a
#    leakage-free Tc value to plug in here.
# -----------------------------------------------------------------------
Kb = 1.380649e-23      # J/K
Pi = np.pi
Phi0 = 2.067833848e-15  # Wb
h_cut = 1.054571817e-34  # J*s

vF = df['vF_predicted_ms'].values
energy_gap = 1.764 * Kb * df['Tc_proxy'].values

df['coherence_length'] = (h_cut * vF) / (Pi * energy_gap)            # metres
df['upper_critical_field'] = Phi0 / (2 * Pi * df['coherence_length'] ** 2)  # Tesla

print(f"\nGL features recomputed using Tc_proxy (leakage-free):")
print(f"  xi  range : {df['coherence_length'].min():.3e} - "
      f"{df['coherence_length'].max():.3e} m")
print(f"  Hc2 range : {df['upper_critical_field'].min():.3f} - "
      f"{df['upper_critical_field'].max():.3f} T")

# NOTE: df['critical_temp'] is left untouched -- it is still the real,
# measured Tc, and is still the correct y_true to score predictions
# against in cell 9. Only the INPUT features (xi, Hc2) now come from a
# leakage-free source. Do not overwrite critical_temp with Tc_proxy.

Tc-proxy compositional features used: 27

Tc_proxy vs real Tc on Stanev (informational, NOT leakage-safe if used for anything beyond reporting):
  R^2 : 0.8058
  MAE : 8.2123 K

GL features recomputed using Tc_proxy (leakage-free):
  xi  range : 9.710e-10 - 2.481e-04 m
  Hc2 range : 0.000 - 349.022 T


In [5]:

# =============================================================================
# 6. COMPUTE GL FEATURES
# =============================================================================
Kb    = 1.380649e-23      # J/K
Pi    = np.pi
Phi0  = 2.067833848e-15   # Wb
h_cut = 1.054571817e-34   # J·s

Tc = np.clip(df['critical_temp'].values, 0.01, None)
vF = df['vF_predicted_ms'].values

energy_gap = 1.764 * Kb * Tc
df['coherence_length']     = (h_cut * vF) / (Pi * energy_gap)   # metres
df['upper_critical_field'] = Phi0 / (2 * Pi * df['coherence_length'] ** 2)  # Tesla

print(f"\nGL features computed:")
print(f"  xi  range : {df['coherence_length'].min():.3e} – "
      f"{df['coherence_length'].max():.3e} m")
print(f"  Hc2 range : {df['upper_critical_field'].min():.3f} – "
      f"{df['upper_critical_field'].max():.3f} T")

# =============================================================================
# 7. SANITY CHECK — family-level vF summary
# =============================================================================
print("\nPer-family vF summary:")
summary = df.groupby('family')['vF_predicted_ms'].agg(
    n='count', mean='mean', std='std', min='min', max='max'
).round(0)
print(summary.to_string())




GL features computed:
  xi  range : 6.158e-10 – 1.240e-04 m
  Hc2 range : 0.000 – 867.939 T

Per-family vF summary:
                         n       mean       std     min      max
family                                                          
Chalcogenide            19    95000.0       0.0   95000    95000
Conventional / Alloys  119  1141978.0  635706.0  152000  1800000
Cuprates                71   270868.0   26172.0  230000   320000
Heavy Fermion            5     7076.0    1161.0    5000     7595
Iron-based              30    85448.0   42607.0   13700   152000
Mg-B related             6   560000.0       0.0  560000   560000


In [6]:

# =============================================================================
# 8. SAVE OUTPUT
# =============================================================================
df.to_csv('stanev_250_new.csv', index=False)
print("\nSaved -> stanev_250_new.csv")
print(f"Final shape: {df.shape}")
print("\nDone.")


Saved -> stanev_250_new.csv
Final shape: (250, 90)

Done.


In [7]:
import pandas as pd
import numpy as np

# Load 5377 unseen compounds file
df_250 = pd.read_csv('stanev_250_new.csv') 

print("250 COMPOUND critical_temp DISTRIBUTION:")
bins = [(0,10),(10,30),(30,50),(50,80),(80,100),(100,135), (135,160), (160,200), (200,250), (250,300)]
for lo,hi in bins:
    n = ((df_250['critical_temp']>=lo)&(df_250['critical_temp']<hi)).sum()
    pct = 100*n/len(df_250)
    print(f"  {lo}-{hi}K: {n} ({pct:.1f}%)")
print(f"\nMean critical_temp: {df_250['critical_temp'].mean():.2f} K")
print(f"Max critical_temp:  {df_250['critical_temp'].max():.2f} K")
print(f"Std critical_temp:  {df_250['critical_temp'].std():.2f} K")

250 COMPOUND critical_temp DISTRIBUTION:
  0-10K: 133 (53.2%)
  10-30K: 46 (18.4%)
  30-50K: 26 (10.4%)
  50-80K: 18 (7.2%)
  80-100K: 11 (4.4%)
  100-135K: 13 (5.2%)
  135-160K: 3 (1.2%)
  160-200K: 0 (0.0%)
  200-250K: 0 (0.0%)
  250-300K: 0 (0.0%)

Mean critical_temp: 25.54 K
Max critical_temp:  150.00 K
Std critical_temp:  34.21 K


In [8]:
import numpy as np
import pandas as pd
import joblib

# # 1. Load my saved model and scaler
my_model = joblib.load("stacking_model_30features.pkl")
scaler     = joblib.load("robust_scaler.pkl")

# 2. Get my top 30 feature names again (just to be safe)
df_top30 = pd.read_csv("top30_vF_xi_Hc_train_83.csv")
top30_cols = [c for c in df_top30.columns if c != "critical_temp"]
print(f"Top 30 features: {top30_cols}")
# 3. Handle NaNs the EXACT same way we did in training (fillna with 0)
# We fill NaNs before converting to .values so Pandas handles it cleanly

df_250 = pd.read_csv('stanev_250_new.csv') 
print(f"External dataset shape before fillna: {df_250.shape}")
df_external_clean = df_250.copy()
print(f"External dataset shape after copy: {df_external_clean.columns}")
df_external_clean[top30_cols] = df_external_clean[top30_cols].fillna(0)
print(f"External dataset shape after fillna: {df_external_clean.shape}")

# 4. Prepare features and target
X_ext = df_external_clean[top30_cols].values
y_ext = df_external_clean["critical_temp"].values

# 5. Scale using my scaler (using .transform() only!)
X_ext_scaled = scaler.transform(X_ext)

# 6. Predict!
y_pred = my_model.predict(X_ext_scaled)

# print the final metrics
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import numpy as np

r2 = r2_score(y_ext, y_pred)
rmse = np.sqrt(mean_squared_error(y_ext, y_pred))
mae = mean_absolute_error(y_ext, y_pred)

print(f"--- FINAL TRUE EXTERNAL VALIDATION ---")
print(f"Compounds tested: {len(y_ext)}")
print(f"R²:   {r2:.4f}")
print(f"RMSE: {rmse:.4f} K")
print(f"MAE:  {mae:.4f} K")


Top 30 features: ['coherence_length', 'upper_critical_field', 'range_ThermalConductivity', 'vF_predicted_ms', 'wtd_mean_Valence', 'wtd_std_ThermalConductivity', 'wtd_gmean_Valence', 'std_atomic_mass', 'range_atomic_radius', 'wtd_gmean_ThermalConductivity', 'wtd_entropy_ThermalConductivity', 'gmean_Density', 'wtd_std_ElectronAffinity', 'wtd_mean_Density', 'wtd_std_Valence', 'wtd_mean_ThermalConductivity', 'mean_Density', 'wtd_std_atomic_radius', 'wtd_entropy_FusionHeat', 'wtd_entropy_Density', 'wtd_entropy_atomic_mass', 'wtd_std_fie', 'wtd_gmean_ElectronAffinity', 'std_ThermalConductivity', 'wtd_gmean_atomic_mass', 'wtd_mean_ElectronAffinity', 'std_atomic_radius', 'wtd_std_Density', 'wtd_std_atomic_mass', 'std_ElectronAffinity']
External dataset shape before fillna: (250, 90)
External dataset shape after copy: Index(['material', 'critical_temp', 'mean_atomic_mass', 'wtd_mean_atomic_mass', 'gmean_atomic_mass', 'wtd_gmean_atomic_mass', 'entropy_atomic_mass', 'wtd_entropy_atomic_mass', 'ra